# 10.1 - PydanticAI

## O que este notebook faz

Este notebook monta um agente `Text-to-SQL` para a base analítica do campo Volve. A proposta é receber uma pergunta em linguagem natural, converter essa pergunta em SQL compatível com SQLite, executar a consulta com guardrails locais e devolver uma resposta curta em português para uso operacional.

### Fluxo completo

1. Define um dicionário de dados em Markdown e o converte para uma estrutura Python reutilizável.
2. Configura os modelos remotos usados pelo `PydanticAI` para duas tarefas separadas:
   - gerar SQL estruturado;
   - sintetizar a resposta final para o operador.
3. Localiza a base SQLite do exercício no filesystem e abre a conexão em modo somente leitura.
4. Lê o schema real da tabela para montar contexto dinâmico do prompt e validar nomes de colunas.
5. Normaliza perguntas e SQL gerado para corrigir ambiguidades comuns, como:
   - nomes curtos de poço;
   - filtros de data em `DATEPRD`;
   - aliases incorretos para volumes de óleo, gás e água.
6. Tenta primeiro um caminho heurístico e determinístico (`fast path`) para perguntas mais simples e frequentes.
7. Quando o `fast path` não resolve, chama o modelo remoto com:
   - schema real;
   - política de escolha de colunas;
   - few-shots selecionados dinamicamente;
   - contexto semântico relevante do dicionário de dados.
8. Valida o SQL antes de executar:
   - bloqueia comandos de escrita;
   - bloqueia previsões implausíveis acima de `D+1`;
   - verifica sintaxe SQLite;
   - tenta reparar identificadores inexistentes usando o schema real.
9. Executa a consulta no SQLite, formata o resultado e monta contexto semântico das colunas retornadas.
10. Envia os dados retornados para um segundo agente, que produz a resposta final em linguagem operacional.
11. Mede tempos, registra logs intermediários e mantém o estado completo do pipeline para inspeção didática.

### Componentes principais

- `AgentState`: estado compartilhado entre os nós do `LangGraph`.
- `try_build_rule_based_sql(...)`: gera SQL sem LLM para perguntas mais previsíveis.
- `build_sql_prompt(...)`: constrói o prompt principal de geração SQL.
- `generate_sql_node(...)`: escolhe entre heurística local e LLM.
- `execute_sql_node(...)`: valida e executa o SQL no banco.
- `respond_node(...)`: transforma o resultado tabular em resposta final.
- `build_workflow_app(...)`: monta o grafo `generate_sql -> execute_sql -> respond`.

### Guardrails e cuidados de segurança

- O banco é aberto em modo `read only`.
- O SQL aceito é somente `SELECT`.
- Projeções diretas acima de `D+1` são bloqueadas.
- Colunas inexistentes são detectadas antes da execução.
- O notebook falha cedo se a chave remota ou o banco não estiverem disponíveis.

### Variáveis de ambiente no terminal

As variáveis abaixo precisam estar definidas no computador/terminal antes da execução, de acordo com o modo de uso do notebook:

- `OPENROUTER_API_KEY`: obrigatória para a chamada dos modelos remotos. Sem ela, o pipeline falha na validação antes da execução.
- `LANGCHAIN_TRACING_V2`: opcional. Use `true` para enviar traces ao LangSmith ou `false` para desligar o tracing.
- `LANGCHAIN_ENDPOINT`: necessária quando `LANGCHAIN_TRACING_V2=true`. Define o endpoint de tracing.
- `LANGCHAIN_API_KEY`: necessária quando `LANGCHAIN_TRACING_V2=true`. Autoriza o envio dos traces.
- `LANGCHAIN_PROJECT`: opcional, mas recomendada quando o tracing estiver ativo para organizar os rastros do projeto.
- `NOTEBOOK_10_1_AUTO_RUN`: opcional. Use `1` para executar automaticamente o pipeline ao final da célula principal ou `0` para estudar o notebook em modo manual.
- `NOTEBOOK_09_AUTO_RUN`: apenas fallback de compatibilidade com versões anteriores do material. Se ambas existirem, `NOTEBOOK_10_1_AUTO_RUN` deve ser a referência principal.

### O que aparece como saída

- pergunta sorteada ou informada;
- prompt de geração SQL;
- SQL final gerado;
- resultado da consulta;
- prompt de resposta;
- resposta final para o operador;
- métricas simples de execução.

### Como usar

- Defina `OPENROUTER_API_KEY` antes de rodar o fluxo remoto.
- Se quiser tracing, configure também `LANGCHAIN_TRACING_V2`, `LANGCHAIN_ENDPOINT`, `LANGCHAIN_API_KEY` e `LANGCHAIN_PROJECT`.
- Use `NOTEBOOK_10_1_AUTO_RUN=0` para estudar o notebook sem disparar a execução automática.
- Se o modo automático estiver ativo, o notebook escolhe uma pergunta do banco de testes e executa o pipeline completo ao final da célula principal.


In [5]:
# Este texto é mantido em Markdown porque a fonte original do exercício já
# descreve o schema nesse formato. Em vez de reescrever manualmente cada campo,
# o notebook reaproveita essa tabela e a converte para um dicionário Python
# logo abaixo.
DATA_DICTIONARY = """
### Dicionário de Dados — Base com Feature Engineering Temporal

| Coluna | Descrição | Tipo de Dados | Unidade de Medida | Equipamento de Medição | Natureza da Variável | Local da Medição |
|---|---|---|---|---|---|---|
| `DATEPRD` | Data da produção/operação diária | datetime | data | Historian | tempo | N/A |
| `WELL_BORE_CODE` | Rótulo operacional expandido do poço (ex.: 'NO 15/9-F-5 AH'); não é o identificador curto usado nas perguntas | string | N/A | Sistema de cadastro corporativo | identificação | N/A |
| `NPD_WELL_BORE_CODE` | Código oficial do poço na NPD | inteiro | N/A | Sistema de cadastro corporativo | identificação | N/A |
| `NPD_WELL_BORE_NAME` | Identificador textual curto do poço na NPD (ex.: '15/9-F-5'); use este campo quando a pergunta citar o código humano do poço | string | N/A | Sistema de cadastro corporativo | identificação | N/A |
| `NPD_FIELD_CODE` | Código oficial do campo | inteiro | N/A | Sistema de cadastro corporativo | identificação | N/A |
| `NPD_FIELD_NAME` | Nome do campo petrolífero | string | N/A | Sistema de cadastro corporativo | identificação | N/A |
| `NPD_FACILITY_CODE` | Código da instalação offshore | inteiro | N/A | Sistema de cadastro corporativo | identificação | N/A |
| `NPD_FACILITY_NAME` | Nome da instalação/FPSO/plataforma | string | N/A | Sistema de cadastro corporativo | identificação | N/A |
| `ON_STREAM_HRS` | Horas em operação no dia | float | horas | Sistema supervisório | tempo | Poço / sistema de produção |
| `AVG_DOWNHOLE_PRESSURE` | Pressão média no fundo do poço | float | bar(a) | Gauge de fundo | pressão | Fundo do poço |
| `AVG_DOWNHOLE_TEMPERATURE` | Temperatura média no fundo do poço | float | °C | Sensor downhole | temperatura | Fundo do poço |
| `AVG_DP_TUBING` | Delta de pressão médio no tubing | float | bar | Sensor de pressão do tubing | pressão | Tubing de produção |
| `AVG_ANNULUS_PRESS` | Pressão média do anular | float | bar | Sensor do anular | pressão | Espaço anular |
| `AVG_CHOKE_SIZE_P` | Abertura média do choke | float | % | Sensor do choke | abertura | Choke de superfície |
| `AVG_CHOKE_UOM` | Unidade de medida do choke | string | texto | Sensor do choke | unidade de abertura | Choke de superfície |
| `AVG_WHP_P` | Pressão média na cabeça do poço | float | bar | Sensor wellhead | pressão | Cabeça do poço |
| `AVG_WHT_P` | Temperatura média na cabeça do poço | float | °C | Sensor wellhead | temperatura | Cabeça do poço |
| `DP_CHOKE_SIZE` | Delta de pressão associado ao choke | float | bar | Sensores do choke | pressão | Linha do choke |
| `BORE_OIL_VOL` | Volume diário de óleo produzido | float | Sm3/d | Medidor multifásico | vazão | Linha de produção / separador |
| `BORE_GAS_VOL` | Volume diário de gás produzido | float | Sm3/d | Medidor de gás | vazão | Linha de gás / separador |
| `BORE_WAT_VOL` | Volume diário de água produzida | float | Sm3/d | Medidor multifásico | vazão | Linha de produção / separador |
| `BORE_WI_VOL` | Volume diário de água injetada | float | Sm3/d | Medidor de injeção | vazão | Linha de injeção de água |
| `FLOW_KIND` | Tipo de fluxo/operação do poço | string | N/A | Sistema supervisório | classificação operacional | N/A |
| `WELL_TYPE` | Tipo do poço | string | N/A | Sistema de engenharia de produção | classificação operacional | N/A |
| `diff_dias` | Diferença de dias entre registros consecutivos | float | dias | Historian | tempo | N/A |
| `oil_lag_1` | Volume de óleo deslocado em 1 período anterior | float | Sm3/d | Medidor multifásico | vazão | Linha de produção / separador |
| `gas_lag_1` | Volume de gás deslocado em 1 período anterior | float | Sm3/d | Medidor de gás | vazão | Linha de gás / separador |
| `water_lag_1` | Volume de água deslocado em 1 período anterior | float | Sm3/d | Medidor multifásico | vazão | Linha de produção / separador |
| `oil_lag_3` | Volume de óleo deslocado em 3 períodos anteriores | float | Sm3/d | Medidor multifásico | vazão | Linha de produção / separador |
| `gas_lag_3` | Volume de gás deslocado em 3 períodos anteriores | float | Sm3/d | Medidor de gás | vazão | Linha de gás / separador |
| `water_lag_3` | Volume de água deslocado em 3 períodos anteriores | float | Sm3/d | Medidor multifásico | vazão | Linha de produção / separador |
| `oil_lag_7` | Volume de óleo deslocado em 7 períodos anteriores | float | Sm3/d | Medidor multifásico | vazão | Linha de produção / separador |
| `gas_lag_7` | Volume de gás deslocado em 7 períodos anteriores | float | Sm3/d | Medidor de gás | vazão | Linha de gás / separador |
| `water_lag_7` | Volume de água deslocado em 7 períodos anteriores | float | Sm3/d | Medidor multifásico | vazão | Linha de produção / separador |
| `oil_lag_14` | Volume de óleo deslocado em 14 períodos anteriores | float | Sm3/d | Medidor multifásico | vazão | Linha de produção / separador |
| `gas_lag_14` | Volume de gás deslocado em 14 períodos anteriores | float | Sm3/d | Medidor de gás | vazão | Linha de gás / separador |
| `water_lag_14` | Volume de água deslocado em 14 períodos anteriores | float | Sm3/d | Medidor multifásico | vazão | Linha de produção / separador |
| `oil_lag_30` | Volume de óleo deslocado em 30 períodos anteriores | float | Sm3/d | Medidor multifásico | vazão | Linha de produção / separador |
| `gas_lag_30` | Volume de gás deslocado em 30 períodos anteriores | float | Sm3/d | Medidor de gás | vazão | Linha de gás / separador |
| `water_lag_30` | Volume de água deslocado em 30 períodos anteriores | float | Sm3/d | Medidor multifásico | vazão | Linha de produção / separador |
| `oil_roll_mean_3` | Média móvel de óleo em janela de 3 períodos | float | Sm3/d | Medidor multifásico | vazão | Linha de produção / separador |
| `gas_roll_mean_3` | Média móvel de gás em janela de 3 períodos | float | Sm3/d | Medidor de gás | vazão | Linha de gás / separador |
| `water_roll_mean_3` | Média móvel de água em janela de 3 períodos | float | Sm3/d | Medidor multifásico | vazão | Linha de produção / separador |
| `oil_roll_mean_7` | Média móvel de óleo em janela de 7 períodos | float | Sm3/d | Medidor multifásico | vazão | Linha de produção / separador |
| `gas_roll_mean_7` | Média móvel de gás em janela de 7 períodos | float | Sm3/d | Medidor de gás | vazão | Linha de gás / separador |
| `water_roll_mean_7` | Média móvel de água em janela de 7 períodos | float | Sm3/d | Medidor multifásico | vazão | Linha de produção / separador |
| `oil_roll_mean_14` | Média móvel de óleo em janela de 14 períodos | float | Sm3/d | Medidor multifásico | vazão | Linha de produção / separador |
| `gas_roll_mean_14` | Média móvel de gás em janela de 14 períodos | float | Sm3/d | Medidor de gás | vazão | Linha de gás / separador |
| `water_roll_mean_14` | Média móvel de água em janela de 14 períodos | float | Sm3/d | Medidor multifásico | vazão | Linha de produção / separador |
| `oil_roll_mean_30` | Média móvel de óleo em janela de 30 períodos | float | Sm3/d | Medidor multifásico | vazão | Linha de produção / separador |
| `gas_roll_mean_30` | Média móvel de gás em janela de 30 períodos | float | Sm3/d | Medidor de gás | vazão | Linha de gás / separador |
| `water_roll_mean_30` | Média móvel de água em janela de 30 períodos | float | Sm3/d | Medidor multifásico | vazão | Linha de produção / separador |
| `oil_roll_std_7` | Desvio padrão móvel de óleo em 7 períodos | float | Sm3/d | Medidor multifásico | dispersão | Linha de produção / separador |
| `gas_roll_std_7` | Desvio padrão móvel de gás em 7 períodos | float | Sm3/d | Medidor de gás | dispersão | Linha de gás / separador |
| `water_roll_std_7` | Desvio padrão móvel de água em 7 períodos | float | Sm3/d | Medidor multifásico | dispersão | Linha de produção / separador |
| `oil_roll_std_14` | Desvio padrão móvel de óleo em 14 períodos | float | Sm3/d | Medidor multifásico | dispersão | Linha de produção / separador |
| `gas_roll_std_14` | Desvio padrão móvel de gás em 14 períodos | float | Sm3/d | Medidor de gás | dispersão | Linha de gás / separador |
| `water_roll_std_14` | Desvio padrão móvel de água em 14 períodos | float | Sm3/d | Medidor multifásico | dispersão | Linha de produção / separador |
| `oil_roll_std_30` | Desvio padrão móvel de óleo em 30 períodos | float | Sm3/d | Medidor multifásico | dispersão | Linha de produção / separador |
| `gas_roll_std_30` | Desvio padrão móvel de gás em 30 períodos | float | Sm3/d | Medidor de gás | dispersão | Linha de gás / separador |
| `water_roll_std_30` | Desvio padrão móvel de água em 30 períodos | float | Sm3/d | Medidor multifásico | dispersão | Linha de produção / separador |
| `oil_delta_1d` | Variação absoluta do óleo em 1 dia | float | Sm3/d | Medidor multifásico | vazão | Linha de produção / separador |
| `gas_delta_1d` | Variação absoluta do gás em 1 dia | float | Sm3/d | Medidor de gás | vazão | Linha de gás / separador |
| `water_delta_1d` | Variação absoluta da água em 1 dia | float | Sm3/d | Medidor multifásico | vazão | Linha de produção / separador |
| `oil_delta_3d` | Variação absoluta do óleo em 3 dias | float | Sm3/d | Medidor multifásico | vazão | Linha de produção / separador |
| `gas_delta_3d` | Variação absoluta do gás em 3 dias | float | Sm3/d | Medidor de gás | vazão | Linha de gás / separador |
| `water_delta_3d` | Variação absoluta da água em 3 dias | float | Sm3/d | Medidor multifásico | vazão | Linha de produção / separador |
| `oil_delta_7d` | Variação absoluta do óleo em 7 dias | float | Sm3/d | Medidor multifásico | vazão | Linha de produção / separador |
| `gas_delta_7d` | Variação absoluta do gás em 7 dias | float | Sm3/d | Medidor de gás | vazão | Linha de gás / separador |
| `water_delta_7d` | Variação absoluta da água em 7 dias | float | Sm3/d | Medidor multifásico | vazão | Linha de produção / separador |
| `oil_pct_change_1d` | Variação percentual do óleo em 1 dia | float | % | Medidor multifásico | variação percentual | Linha de produção / separador |
| `gas_pct_change_1d` | Variação percentual do gás em 1 dia | float | % | Medidor de gás | variação percentual | Linha de gás / separador |
| `water_pct_change_1d` | Variação percentual da água em 1 dia | float | % | Medidor multifásico | variação percentual | Linha de produção / separador |
| `oil_pct_change_7d` | Variação percentual do óleo em 7 dias | float | % | Medidor multifásico | variação percentual | Linha de produção / separador |
| `gas_pct_change_7d` | Variação percentual do gás em 7 dias | float | % | Medidor de gás | variação percentual | Linha de gás / separador |
| `water_pct_change_7d` | Variação percentual da água em 7 dias | float | % | Medidor multifásico | variação percentual | Linha de produção / separador |
| `oil_pct_change_14d` | Variação percentual do óleo em 14 dias | float | % | Medidor multifásico | variação percentual | Linha de produção / separador |
| `gas_pct_change_14d` | Variação percentual do gás em 14 dias | float | % | Medidor de gás | variação percentual | Linha de gás / separador |
| `water_pct_change_14d` | Variação percentual da água em 14 dias | float | % | Medidor multifásico | variação percentual | Linha de produção / separador |
| `oil_ewma_3` | Média móvel exponencial do óleo com janela 3 | float | Sm3/d | Medidor multifásico | vazão | Linha de produção / separador |
| `gas_ewma_3` | Média móvel exponencial do gás com janela 3 | float | Sm3/d | Medidor de gás | vazão | Linha de gás / separador |
| `water_ewma_3` | Média móvel exponencial da água com janela 3 | float | Sm3/d | Medidor multifásico | vazão | Linha de produção / separador |
| `oil_ewma_7` | Média móvel exponencial do óleo com janela 7 | float | Sm3/d | Medidor multifásico | vazão | Linha de produção / separador |
| `gas_ewma_7` | Média móvel exponencial do gás com janela 7 | float | Sm3/d | Medidor de gás | vazão | Linha de gás / separador |
| `water_ewma_7` | Média móvel exponencial da água com janela 7 | float | Sm3/d | Medidor multifásico | vazão | Linha de produção / separador |
| `oil_ewma_14` | Média móvel exponencial do óleo com janela 14 | float | Sm3/d | Medidor multifásico | vazão | Linha de produção / separador |
| `gas_ewma_14` | Média móvel exponencial do gás com janela 14 | float | Sm3/d | Medidor de gás | vazão | Linha de gás / separador |
| `water_ewma_14` | Média móvel exponencial da água com janela 14 | float | Sm3/d | Medidor multifásico | vazão | Linha de produção / separador |
| `oil_ewma_30` | Média móvel exponencial do óleo com janela 30 | float | Sm3/d | Medidor multifásico | vazão | Linha de produção / separador |
| `gas_ewma_30` | Média móvel exponencial do gás com janela 30 | float | Sm3/d | Medidor de gás | vazão | Linha de gás / separador |
| `water_ewma_30` | Média móvel exponencial da água com janela 30 | float | Sm3/d | Medidor multifásico | vazão | Linha de produção / separador |
| `oil_expanding_mean` | Média acumulada expandida do óleo | float | Sm3/d | Medidor multifásico | vazão | Linha de produção / separador |
| `oil_expanding_std` | Desvio padrão acumulado expandido do óleo | float | Sm3/d | Medidor multifásico | dispersão | Linha de produção / separador |
| `gas_expanding_mean` | Média acumulada expandida do gás | float | Sm3/d | Medidor de gás | vazão | Linha de gás / separador |
| `gas_expanding_std` | Desvio padrão acumulado expandido do gás | float | Sm3/d | Medidor de gás | dispersão | Linha de gás / separador |
| `water_expanding_mean` | Média acumulada expandida da água | float | Sm3/d | Medidor multifásico | vazão | Linha de produção / separador |
| `water_expanding_std` | Desvio padrão acumulado expandido da água | float | Sm3/d | Medidor multifásico | dispersão | Linha de produção / separador |
| `oil_cumulative` | Volume acumulado de óleo ao longo do tempo | float | Sm3 | Medidor multifásico | volume | Linha de produção / separador |
| `gas_cumulative` | Volume acumulado de gás ao longo do tempo | float | Sm3 | Medidor de gás | volume | Linha de gás / separador |
| `water_cumulative` | Volume acumulado de água ao longo do tempo | float | Sm3 | Medidor multifásico | volume | Linha de produção / separador |
| `oil_velocity` | Velocidade de variação do óleo entre períodos | float | Sm3/dia | Medidor multifásico | velocidade | Linha de produção / separador |
| `oil_acceleration` | Aceleração da variação do óleo entre períodos | float | Sm3/dia² | Medidor multifásico | aceleração | Linha de produção / separador |
| `gas_velocity` | Velocidade de variação do gás entre períodos | float | Sm3/dia | Medidor de gás | velocidade | Linha de gás / separador |
| `gas_acceleration` | Aceleração da variação do gás entre períodos | float | Sm3/dia² | Medidor de gás | aceleração | Linha de gás / separador |
| `oil_trend_strength` | Intensidade da tendência do óleo frente à média móvel | float | adimensional | Medidor multifásico | tendência | Linha de produção / separador |
| `gas_trend_strength` | Intensidade da tendência do gás frente à média móvel | float | adimensional | Medidor de gás | tendência | Linha de gás / separador |
| `water_trend_strength` | Intensidade da tendência da água frente à média móvel | float | adimensional | Medidor multifásico | tendência | Linha de produção / separador |
| `oil_vs_trend` | Razão entre óleo observado e tendência estimada | float | adimensional | Medidor multifásico | tendência | Linha de produção / separador |
| `gas_vs_trend` | Razão entre gás observado e tendência estimada | float | adimensional | Medidor de gás | tendência | Linha de gás / separador |
| `water_vs_trend` | Razão entre água observada e tendência estimada | float | adimensional | Medidor multifásico | tendência | Linha de produção / separador |
| `oil_volatility_index` | Índice relativo de volatilidade do óleo | float | adimensional | Medidor multifásico | volatilidade | Linha de produção / separador |
| `gas_volatility_index` | Índice relativo de volatilidade do gás | float | adimensional | Medidor de gás | volatilidade | Linha de gás / separador |
| `oil_momentum_7d` | Momentum do óleo em 7 dias | float | Sm3/d | Medidor multifásico | momentum | Linha de produção / separador |
| `oil_momentum_30d` | Momentum do óleo em 30 dias | float | Sm3/d | Medidor multifásico | momentum | Linha de produção / separador |
| `oil_roc_7d` | Taxa de variação do óleo em 7 dias | float | % | Medidor multifásico | variação percentual | Linha de produção / separador |
| `oil_roc_30d` | Taxa de variação do óleo em 30 dias | float | % | Medidor multifásico | variação percentual | Linha de produção / separador |
| `oil_zscore_30` | Z-score do óleo em janela de 30 períodos | float | adimensional | Medidor multifásico | desvio padronizado | Linha de produção / separador |
""".strip()


def parse_markdown_data_dictionary(markdown_text):
    # Converte a tabela Markdown em um dicionário indexado pelo nome da coluna.
    # Isso permite reaproveitar o mesmo conteúdo tanto na documentação quanto
    # na geração de contexto para o LLM.
    data_dictionary = {}

    for raw_line in markdown_text.splitlines():
        line = raw_line.strip()
        if not line.startswith("|"):
            continue
        # Ignora cabeçalho e linha separadora da tabela Markdown.
        if line.startswith("| Coluna |") or line.startswith("|---"):
            continue

        row_parts = [part.strip() for part in line.split("|")[1:-1]]
        if len(row_parts) != 7:
            continue

        (
            column_name,
            descricao,
            tipo,
            unidade,
            equipamento,
            natureza,
            local,
        ) = row_parts
        column_name = column_name.strip("`")

        data_dictionary[column_name] = {
            "descricao": descricao,
            "tipo": tipo,
            "unidade": unidade,
            "equipamento": equipamento,
            "natureza": natureza,
            "local": local,
        }

    return data_dictionary

# Estrutura final usada pelo restante do notebook para enriquecer prompts,
# explicar colunas retornadas e reduzir alucinação de schema.
COLUMN_DATA_DICTIONARY = parse_markdown_data_dictionary(DATA_DICTIONARY)


In [6]:
import json
import os
import re
import time
import sqlite3
import numpy as np
import pandas as pd
import sqlglot
from langgraph.graph import END, START, StateGraph
from pydantic import BaseModel, Field
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider
from textwrap import dedent
from difflib import SequenceMatcher
from typing import Dict, Any, List, Tuple
from typing_extensions import TypedDict

# =============================================================================
# VISÃO GERAL DO NOTEBOOK
# -----------------------------------------------------------------------------
# Este notebook implementa um agente Text-to-SQL para o caso Volve.
#
# A ideia central é separar o problema em 3 etapas:
# 1. Entender a pergunta em linguagem natural e gerar SQL.
# 2. Executar esse SQL com segurança no SQLite.
# 3. Transformar o resultado tabular em uma resposta operacional curta.
#
# Em outras palavras:
# pergunta humana -> SQL -> dados reais -> resposta final
#
# O notebook também mede tempo e tamanho de contexto, porque em sistemas com
# LLM isso é parte do custo e da qualidade, não apenas detalhe de engenharia.
# =============================================================================

# O AgentState é a "memória de curto prazo" do grafo.
# Cada nó do LangGraph lê e escreve algumas chaves desse dicionário.
class AgentState(TypedDict, total=False):
    question: str
    generated_sql: str
    error_message: str
    retry_count: int
    db_data: List[Dict[str, Any]]
    query_result: str
    query_column_context: str
    sql_generation_time: float
    sql_execution_time: float
    response_generation_time: float
    remote_response_time: float
    local_prompt_chars: int
    remote_prompt_chars: int
    final_answer: str
    final_response: str
    alerta_critico: bool

MAX_SQL_RETRIES = 3
DB_NAME = "volve_with_feature_engineering_temporal.db"
TABLE_NAME = "volve_with_feature_engineering_temporal"

# -----------------------------------------------------------------------------
# CONFIGURACAO RAPIDA DE MODELOS REMOTOS PARA TESTES
# Todo o notebook usa apenas estas duas configuracoes.
# -----------------------------------------------------------------------------
# REMOTE_SQL_MODEL = "deepseek/deepseek-chat"
REMOTE_SQL_MODEL = "google/gemini-2.5-flash"
REMOTE_TEXT_MODEL = "google/gemini-2.5-flash"

# Variáveis de tracing do ecossistema LangChain/LangSmith.
# São opcionais, mas quando o tracing estiver ligado precisam estar coerentes
# no ambiente do terminal para o rastreamento funcionar corretamente.
LANGCHAIN_TRACING_V2 = os.getenv("LANGCHAIN_TRACING_V2", "true")
LANGCHAIN_ENDPOINT = os.getenv("LANGCHAIN_ENDPOINT", "https://api.smith.langchain.com")
LANGCHAIN_API_KEY = os.getenv("LANGCHAIN_API_KEY", "")
LANGCHAIN_PROJECT = os.getenv("LANGCHAIN_PROJECT", "lab-artificial-inteligence")

os.environ["LANGCHAIN_TRACING_V2"] = LANGCHAIN_TRACING_V2
if LANGCHAIN_TRACING_V2.strip().lower() == "true":
    os.environ["LANGCHAIN_ENDPOINT"] = LANGCHAIN_ENDPOINT
    os.environ["LANGCHAIN_API_KEY"] = LANGCHAIN_API_KEY
    os.environ["LANGCHAIN_PROJECT"] = LANGCHAIN_PROJECT
else:
    os.environ.pop("LANGCHAIN_ENDPOINT", None)
    os.environ.pop("LANGCHAIN_API_KEY", None)
    os.environ.pop("LANGCHAIN_PROJECT", None)

# LOCAL_SQL_MODEL = "qwen2.5-coder:14b"
# LOCAL_TEXT_MODEL = "llama3.1:8b"

def raw_log(message: Any) -> None:
    # Função mais "crua" possível para escrever no terminal.
    # Mantemos essa função separada para não misturar formatação com I/O.
    print(str(message), flush=True)

def safe_str(text: Any) -> str:
    # Em integrações com LLM, APIs e DataFrames, é comum receber:
    # - None
    # - bytes estranhos
    # - objetos que precisam virar texto
    #
    # Esta função centraliza a normalização de string para reduzir ruído.
    if text is None:
        return ""

    return str(text).encode("utf-8", errors="ignore").decode("utf-8")

def log_progress(message: str) -> None:
    # Todo log "normal" do pipeline passa por aqui.
    # A função safe_str evita que caracteres ruins quebrem a execução.
    raw_log(safe_str(message))

def build_sql_prompt_for_display(full_prompt: str) -> str:
    prompt_text = safe_str(full_prompt)
    rules_marker = "\nRegras Cruciais:\n"
    question_marker = "\nPergunta: "

    if rules_marker not in prompt_text:
        return prompt_text

    visible_prefix = prompt_text.split(rules_marker, 1)[0].strip()
    question_index = prompt_text.rfind(question_marker)
    if question_index == -1:
        return visible_prefix

    visible_question = prompt_text[question_index + 1:].strip()
    return f"{visible_prefix}\n\n{visible_question}".strip()

def resolve_database_path(db_name: str) -> str:
    # O notebook tenta achar arquivos de dados em mais de um lugar porque,
    # dependendo de onde a célula foi executada, o diretório atual pode mudar.
    current_workdir = os.getcwd()
    notebook_directory = os.path.join(
        current_workdir,
        "notebooks",
        "10-exercicio-production-surveillance",
    )
    candidate_paths = [
        os.path.abspath(db_name),
        os.path.abspath(os.path.join(current_workdir, "notebooks", db_name)),
        os.path.abspath(os.path.join(notebook_directory, db_name)),
    ]

    for candidate_path in candidate_paths:
        if os.path.exists(candidate_path):
            return candidate_path

    # Se nenhum caminho existir, devolvemos o mais específico para a mensagem
    # de erro apontar diretamente para o local esperado do exercício.
    return candidate_paths[-1]

def build_read_only_sqlite_uri(db_path: str) -> str:
    # mode=ro = read only.
    #
    # Mesmo que o LLM alucine um comando destrutivo, a conexão de leitura ajuda
    # a impedir escrita física no banco.
    return f"file:{db_path}?mode=ro"

def clean_generated_sql(raw_sql: str) -> str:
    # Muitos modelos gostam de devolver:
    # ```sql
    # SELECT ...
    # ```
    #
    # Para execução automática isso atrapalha. Aqui limpamos esse excesso.
    cleaned_sql = safe_str(raw_sql)
    for token in ("```sql", "```"):
        cleaned_sql = cleaned_sql.replace(token, "")
    return cleaned_sql.strip().rstrip(";").strip()

SHORT_WELL_NAME_PATTERN = re.compile(r"\b\d+/\d+-[A-Z]-\d+(?: [A-Z])?\b")

def extract_short_well_names_from_question(question: str) -> List[str]:
    # dict.fromkeys preserva a ordem original e remove duplicatas.
    question_text = safe_str(question)
    return list(dict.fromkeys(
        match.group(0)
        for match in SHORT_WELL_NAME_PATTERN.finditer(question_text)
    ))

def enrich_question_with_schema_hints(question: str) -> str:
    normalized_question = safe_str(question).strip()
    short_well_names = extract_short_well_names_from_question(normalized_question)
    if not short_well_names:
        return normalized_question

    rendered_names = ", ".join(f"'{well_name}'" for well_name in short_well_names)
    schema_hint = (
        "HINT DE SCHEMA: quando a pergunta citar identificadores curtos de poço como "
        f"{rendered_names}, filtre pela coluna NPD_WELL_BORE_NAME. "
        "Nao use WELL_BORE_CODE nem NPD_WELL_BORE_CODE para igualdade textual nesses casos, "
        "porque WELL_BORE_CODE contem rotulos expandidos como 'NO 15/9-F-5 AH'."
    )
    return f"{normalized_question}\n{schema_hint}"

def normalize_well_name_filters_in_sql(sql_query: str) -> str:
    # Corrige um erro comum do modelo: usar WELL_BORE_CODE ou
    # NPD_WELL_BORE_CODE para comparar o nome curto humano do poço.
    normalized_sql = safe_str(sql_query)
    equality_pattern = (
        r"(?P<prefix>\b\w+\.)?"
        r"(?:WELL_BORE_CODE|NPD_WELL_BORE_CODE)\s*=\s*"
        r"['\"](?P<value>\d+/\d+-[A-Z]-\d+(?: [A-Z])?)['\"]"
    )
    in_list_pattern = (
        r"(?P<prefix>\b\w+\.)?"
        r"(?:WELL_BORE_CODE|NPD_WELL_BORE_CODE)\s+IN\s*"
        r"\((?P<items>[^)]*)\)"
    )

    def replace_textual_identifier_equals(match: re.Match[str]) -> str:
        prefix = safe_str(match.group("prefix") or "")
        value = safe_str(match.group("value"))
        return f"{prefix}NPD_WELL_BORE_NAME = '{value}'"

    normalized_sql = re.sub(
        equality_pattern,
        replace_textual_identifier_equals,
        normalized_sql,
        flags=re.IGNORECASE,
    )

    def replace_textual_identifier_in(match: re.Match[str]) -> str:
        prefix = safe_str(match.group("prefix") or "")
        items_text = safe_str(match.group("items"))
        items = re.findall(r"['\"]([^'\"]+)['\"]", items_text)
        if not items:
            return match.group(0)
        if not all(SHORT_WELL_NAME_PATTERN.fullmatch(item) for item in items):
            return match.group(0)

        rendered_items = ", ".join(f"'{item}'" for item in items)
        return f"{prefix}NPD_WELL_BORE_NAME IN ({rendered_items})"

    normalized_sql = re.sub(
        in_list_pattern,
        replace_textual_identifier_in,
        normalized_sql,
        flags=re.IGNORECASE,
    )

    return normalized_sql

def normalize_dateprd_filters_in_sql(sql_query: str) -> str:
    # DATEPRD é timestamp. Se a pergunta usa apenas data civil, transformamos
    # a comparação para date(DATEPRD) a fim de evitar falso negativo por hora.
    normalized_sql = safe_str(sql_query)
    date_comparison_pattern = (
        r"(?<!date\()(?P<column>(?:\b\w+\.)?DATEPRD)\s*"
        r"(?P<operator>=|>=|<=|>|<)\s*"
        r"['\"](?P<value>\d{4}-\d{2}-\d{2})['\"]"
    )
    date_between_pattern = (
        r"(?<!date\()(?P<column>(?:\b\w+\.)?DATEPRD)\s+BETWEEN\s+"
        r"['\"](?P<start>\d{4}-\d{2}-\d{2})['\"]\s+AND\s+"
        r"['\"](?P<end>\d{4}-\d{2}-\d{2})['\"]"
    )

    def replace_date_comparison(match: re.Match[str]) -> str:
        column_expr = safe_str(match.group("column"))
        operator = safe_str(match.group("operator"))
        date_value = safe_str(match.group("value"))
        return f"date({column_expr}) {operator} '{date_value}'"

    normalized_sql = re.sub(
        date_comparison_pattern,
        replace_date_comparison,
        normalized_sql,
        flags=re.IGNORECASE,
    )

    def replace_date_between(match: re.Match[str]) -> str:
        column_expr = safe_str(match.group("column"))
        start_date = safe_str(match.group("start"))
        end_date = safe_str(match.group("end"))
        return f"date({column_expr}) BETWEEN '{start_date}' AND '{end_date}'"

    normalized_sql = re.sub(
        date_between_pattern,
        replace_date_between,
        normalized_sql,
        flags=re.IGNORECASE,
    )

    return normalized_sql

SQL_METRIC_SYNONYMS = {
    "WATER_TOTAL_VOL": "BORE_WAT_VOL",
    "WATER_VOL": "BORE_WAT_VOL",
    "OIL_TOTAL_VOL": "BORE_OIL_VOL",
    "OIL_VOL": "BORE_OIL_VOL",
    "GAS_TOTAL_VOL": "BORE_GAS_VOL",
    "GAS_VOL": "BORE_GAS_VOL",
}

def infer_primary_fluid_from_text(text: str) -> str:
    normalized_text = safe_str(text).lower()
    if any(term in normalized_text for term in ("água", "agua", "water", "wat")):
        return "water"
    if any(term in normalized_text for term in ("óleo", "oleo", "oil")):
        return "oil"
    if "gas" in normalized_text or "gás" in normalized_text:
        return "gas"
    return ""

def extract_identifier_tokens(text: str) -> List[str]:
    return [token for token in re.findall(r"[a-z0-9]+", safe_str(text).lower()) if token]

def resolve_volume_alias_identifier(identifier: str) -> str:
    normalized_identifier = safe_str(identifier).lower()
    alias_patterns = {
        "BORE_OIL_VOL": [
            r"^(?:bore_)?oil(?:_total|_production|_prod|_daily)?(?:_vol(?:ume)?)?$",
        ],
        "BORE_GAS_VOL": [
            r"^(?:bore_)?gas(?:_total|_production|_prod|_daily)?(?:_vol(?:ume)?)?$",
        ],
        "BORE_WAT_VOL": [
            r"^(?:bore_)?water(?:_total|_production|_prod|_daily)?(?:_vol(?:ume)?)?$",
            r"^(?:bore_)?wat(?:_total|_production|_prod|_daily)?(?:_vol(?:ume)?)?$",
        ],
        "BORE_WI_VOL": [
            r"^(?:bore_)?water(?:_injection|_inj|_wi)(?:_total|_daily)?(?:_vol(?:ume)?)?$",
            r"^(?:bore_)?wi(?:_vol(?:ume)?)?$",
        ],
    }

    for target_column, patterns in alias_patterns.items():
        for pattern in patterns:
            if re.fullmatch(pattern, normalized_identifier):
                return target_column

    return SQL_METRIC_SYNONYMS.get(safe_str(identifier).upper(), "")

def normalize_metric_aliases_in_sql(sql_query: str) -> str:
    normalized_sql = safe_str(sql_query)
    candidate_identifiers = sorted(set(re.findall(r"\b[A-Za-z_][A-Za-z0-9_]*\b", normalized_sql)))
    for identifier in candidate_identifiers:
        target_name = resolve_volume_alias_identifier(identifier)
        if not target_name:
            continue
        normalized_sql = re.sub(
            rf"\b{re.escape(identifier)}\b",
            target_name,
            normalized_sql,
            flags=re.IGNORECASE,
        )
    return normalized_sql

def rank_schema_column_candidates(identifier: str) -> List[Tuple[float, str]]:
    normalized_identifier = safe_str(identifier).lower()
    identifier_tokens = set(extract_identifier_tokens(normalized_identifier))
    identifier_fluid = infer_primary_fluid_from_text(normalized_identifier)
    ranked_candidates: List[Tuple[float, str]] = []

    for column_name in SCHEMA_COLUMN_NAMES:
        normalized_column = safe_str(column_name).lower()
        column_tokens = set(extract_identifier_tokens(normalized_column))
        score = SequenceMatcher(None, normalized_identifier, normalized_column).ratio()
        score += min(len(identifier_tokens & column_tokens), 3) * 0.08
        if identifier_fluid and infer_primary_fluid_from_text(normalized_column) == identifier_fluid:
            score += 0.12
        if any(token.isdigit() and token in column_tokens for token in identifier_tokens):
            score += 0.05
        if (
            any(token in {"vol", "volume", "total"} for token in identifier_tokens)
            and normalized_column.startswith("bore_")
        ):
            score += 0.08
        ranked_candidates.append((score, column_name))

    ranked_candidates.sort(key=lambda item: item[0], reverse=True)
    return ranked_candidates

def try_resolve_unknown_identifier(identifier: str) -> Tuple[str, float, List[str]]:
    exact_match = SCHEMA_COLUMNS_BY_LOWER.get(safe_str(identifier).lower(), "")
    if exact_match:
        return exact_match, 1.0, [exact_match]

    alias_match = resolve_volume_alias_identifier(identifier)
    if alias_match and alias_match in SCHEMA_TYPES_BY_NAME:
        return alias_match, 1.0, [alias_match]

    ranked_candidates = rank_schema_column_candidates(identifier)
    suggestions = [column_name for score, column_name in ranked_candidates[:5] if score >= 0.45]
    if not ranked_candidates:
        return "", 0.0, suggestions

    best_score, best_column = ranked_candidates[0]
    second_score = ranked_candidates[1][0] if len(ranked_candidates) > 1 else 0.0
    if best_score >= 0.92 or (best_score >= 0.88 and (best_score - second_score) >= 0.12):
        return best_column, best_score, suggestions

    return "", best_score, suggestions

def repair_sql_identifiers_against_schema(sql_query: str) -> Tuple[str, str]:
    # Aqui tentamos corrigir pequenos erros de nomenclatura antes de desistir.
    # Exemplo: WATER_TOTAL_VOL -> BORE_WAT_VOL.
    repaired_sql = safe_str(sql_query)

    try:
        parsed_sql = sqlglot.parse_one(repaired_sql, read="sqlite")
    except sqlglot.errors.ParseError:
        return repaired_sql, ""

    unknown_identifiers = sorted({
        safe_str(column.name)
        for column in parsed_sql.find_all(sqlglot.expressions.Column)
        if safe_str(column.name) and safe_str(column.name).lower() not in SCHEMA_COLUMNS_BY_LOWER
    })

    unresolved_messages: List[str] = []
    for identifier in unknown_identifiers:
        resolved_identifier, confidence, suggestions = try_resolve_unknown_identifier(identifier)
        if resolved_identifier and resolved_identifier.lower() != safe_str(identifier).lower():
            repaired_sql = re.sub(
                rf"\b{re.escape(identifier)}\b",
                resolved_identifier,
                repaired_sql,
                flags=re.IGNORECASE,
            )
            repair_message = (
                "[SQL][REPAIR_SCHEMA_IDENTIFIER] coluna "
                f"'{identifier}' ajustada para '{resolved_identifier}' "
                f"com confidence={confidence:.2f}"
            )
            log_progress(repair_message)
            continue

        if suggestions:
            unresolved_messages.append(
                f"{identifier} -> sugestoes: {', '.join(suggestions[:4])}"
            )
        else:
            unresolved_messages.append(f"{identifier} -> sem sugestoes confiaveis")

    if unresolved_messages:
        error_message = (
            "Validação semântica do schema detectou colunas inexistentes antes da execução: "
            + " | ".join(unresolved_messages)
            + ". Reescreva usando apenas colunas reais do schema."
        )
        return repaired_sql, error_message

    return repaired_sql, ""

def normalize_and_repair_sql(sql_query: str) -> Tuple[str, str]:
    # Este helper centraliza o pós-processamento local do SQL.
    # Ordem aplicada:
    # 1. limpa markdown/acessórios do modelo;
    # 2. corrige aliases de métricas;
    # 3. corrige colunas erradas para nome curto de poço;
    # 4. ajusta filtros de DATEPRD para comparação por data;
    # 5. valida/repara identificadores contra o schema real.
    normalized_sql = clean_generated_sql(sql_query)
    normalized_sql = normalize_metric_aliases_in_sql(normalized_sql)
    normalized_sql = normalize_well_name_filters_in_sql(normalized_sql)
    normalized_sql = normalize_dateprd_filters_in_sql(normalized_sql)
    normalized_sql, schema_error = repair_sql_identifiers_against_schema(normalized_sql)
    return normalized_sql, schema_error

def format_value_for_prompt(column_name: str, value: Any) -> str:
    # Esta função prepara valores para entrar no prompt da resposta final.
    #
    # Objetivo didático importante:
    # o LLM responde melhor quando o dado chega em formato humano.
    # Exemplo:
    # - 0.15129  -> 15.13%
    # - 446.6200 -> 446.62
    if pd.isna(value):
        return "NA"

    normalized_name = safe_str(column_name).lower()

    if isinstance(value, (np.integer, int)):
        return safe_str(int(value))

    if isinstance(value, (np.floating, float)):
        numeric_value = float(value)
        if "pct" in normalized_name or "_roc_" in normalized_name:
            return f"{numeric_value * 100:.2f}%"
        return f"{numeric_value:.2f}"

    return safe_str(value)

def format_dataframe_for_prompt(df: pd.DataFrame) -> str:
    # O DataFrame é copiado para não alterar os dados originais em memória.
    # Em seguida, formatamos coluna por coluna antes de serializar para texto.
    formatted_df = df.copy()
    for column_name in formatted_df.columns:
        formatted_df[column_name] = formatted_df[column_name].map(
            lambda value, col=column_name: format_value_for_prompt(col, value)
        )
    return safe_str(formatted_df.to_string(index=False))

def build_sql_generation_error(
    state: AgentState,
    error_message: str,
    elapsed: float,
    local_prompt_chars: int,
) -> Dict[str, Any]:
    # Em arquiteturas com grafo, erro também é estado.
    #
    # Em vez de simplesmente dar raise, devolvemos um dicionário que o próximo
    # nó consegue interpretar para decidir entre retry ou encerramento.
    return {
        "generated_sql": "",
        "error_message": safe_str(error_message),
        "retry_count": state.get("retry_count", 0) + 1,
        "sql_generation_time": state.get("sql_generation_time", 0.0) + elapsed,
        "local_prompt_chars": state.get("local_prompt_chars", 0) + local_prompt_chars,
    }

def build_empty_execution_response(error_message: str) -> Dict[str, Any]:
    # Esta resposta "vazia" é usada quando a etapa SQL não deve prosseguir.
    # Exemplo: guardrail bloqueou a query.
    return {
        "error_message": safe_str(error_message),
        "db_data": [],
        "query_result": "",
        "query_column_context": "",
    }

def try_build_rule_based_sql(question: str) -> str:
    # Este é um atalho importante:
    # para perguntas muito simples, não vale a pena pagar custo de LLM.
    #
    # Se detectarmos uma pergunta simples de média, máximo ou mínimo usando uma
    # única coluna explícita, respondemos com SQL determinístico.
    normalized_question = safe_str(question).lower()
    explicit_columns = [
        column_name
        for column_name in SCHEMA_COLUMN_NAMES
        if column_name.lower() in normalized_question and column_name != "DATEPRD"
    ]

    semantic_columns = infer_sensor_columns_from_question(question)
    candidate_columns = list(dict.fromkeys(explicit_columns + semantic_columns))

    if len(candidate_columns) != 1:
        return ""

    column_name = candidate_columns[0]
    normalized_words = set(
        word for word in re.sub(r"[^\w]+", " ", normalized_question).split() if word
    )

    avg_tokens = {"media", "média"}
    max_tokens = {"maior", "máxima", "maxima", "máximo", "maximo", "pico"}
    min_tokens = {"menor", "mínima", "minima", "mínimo", "minimo"}
    latest_measurement_phrases = (
        "mais recente",
        "leitura mais recente",
        "leitura registrada",
        "última leitura",
        "ultima leitura",
        "medição mais recente",
        "medicao mais recente",
        "último valor",
        "ultimo valor",
        "valor mais recente",
    )
    short_well_names = extract_short_well_names_from_question(question)
    base_filters: List[str] = []
    if len(short_well_names) == 1:
        base_filters.append(f"NPD_WELL_BORE_NAME = '{short_well_names[0]}'")
    elif len(short_well_names) > 1:
        rendered_names = ", ".join(f"'{well_name}'" for well_name in short_well_names)
        base_filters.append(f"NPD_WELL_BORE_NAME IN ({rendered_names})")

    if column_name in {"ON_STREAM_HRS", "BORE_OIL_VOL", "BORE_GAS_VOL", "BORE_WAT_VOL"}:
        base_filters.append("FLOW_KIND = 'production'")

    non_null_filter = f"{column_name} IS NOT NULL"

    def render_where(extra_filters: List[str] | None = None) -> str:
        filters = list(base_filters)
        if extra_filters:
            filters.extend(extra_filters)
        if not filters:
            return ""
        return f" WHERE {' AND '.join(filters)}"

    if any(phrase in normalized_question for phrase in latest_measurement_phrases):
        return (
            f"SELECT DATEPRD, NPD_WELL_BORE_NAME, {column_name} "
            f"FROM {TABLE_NAME}"
            f"{render_where([non_null_filter])} "
            f"ORDER BY DATEPRD DESC LIMIT 1"
        )

    # Média: devolve um único valor agregado.
    if normalized_words & avg_tokens:
        alias_name = f"avg_{column_name.lower()}"
        return f"SELECT AVG({column_name}) AS {alias_name} FROM {TABLE_NAME}{render_where([non_null_filter])}"

    # Máximo: devolve a data em que o valor foi máximo e o valor.
    if normalized_words & max_tokens:
        return (
            f"SELECT DATEPRD, {column_name} "
            f"FROM {TABLE_NAME} "
            f"{render_where([non_null_filter])} "
            f"ORDER BY {column_name} DESC LIMIT 1"
        )

    # Mínimo: mesma lógica do máximo, invertendo a ordenação.
    if normalized_words & min_tokens:
        return (
            f"SELECT DATEPRD, {column_name} "
            f"FROM {TABLE_NAME} "
            f"{render_where([non_null_filter])} "
            f"ORDER BY {column_name} ASC LIMIT 1"
        )

    return ""

DB_PATH = resolve_database_path(DB_NAME)
if not os.path.exists(DB_PATH):
    raise FileNotFoundError(f"Arquivo SQLite não encontrado: {DB_PATH}")

# Em notebooks, reexecuções da célula podem deixar conexões antigas abertas.
# Por isso tentamos fechar a conexão anterior antes de reabrir a conexão.
if "conn" in globals():
    try:
        conn.close()
    except Exception:
        pass

# O pipeline consulta apenas a base SQLite já materializada.
# Não carregamos o CSV em memória porque ele não participa do fluxo atual.
conn = sqlite3.connect(
    build_read_only_sqlite_uri(DB_PATH),
    uri=True,
    # check_same_thread=False evita restrições desnecessárias em cenários de
    # reuso dentro do notebook e da orquestração.
    check_same_thread=False,
)
schema_df = pd.read_sql(f"PRAGMA table_info({TABLE_NAME})", conn)
# schema_text vira uma representação curta do schema para o prompt SQL.
schema_text = ", ".join(
    f"{safe_str(row['name'])} ({safe_str(row['type'] or 'TEXT')})"
    for _, row in schema_df.iterrows()
)

# SCHEMA_COLUMN_NAMES ajuda em validações, fast path e contexto.
SCHEMA_COLUMN_NAMES = [safe_str(column_name) for column_name in schema_df["name"].tolist()]
SCHEMA_COLUMNS_BY_LOWER = {
    safe_str(column_name).lower(): safe_str(column_name)
    for column_name in SCHEMA_COLUMN_NAMES
}

# SCHEMA_TYPES_BY_NAME ajuda a explicar para o LLM "o que é" cada coluna.
SCHEMA_TYPES_BY_NAME = {
    safe_str(row["name"]): safe_str(row["type"] or "TEXT")
    for _, row in schema_df.iterrows()
}

# Reusa o dicionário global definido na célula anterior.

def infer_metadata_from_column_name(column_name: str) -> Dict[str, str]:
    # Primeiro tentamos descrição explícita do dicionário de dados.
    # Se a coluna for derivada, montamos uma descrição heurística útil para o
    # LLM entender a natureza da métrica mesmo sem documentação manual.
    normalized_name = safe_str(column_name)
    if normalized_name in COLUMN_DATA_DICTIONARY:
        return COLUMN_DATA_DICTIONARY[normalized_name]

    base_info = {
        "descricao": f"Coluna analítica derivada: {normalized_name}",
        "tipo": "float",
        "unidade": "N/A",
        "equipamento": "Dataset analítico",
        "natureza": "analítica",
        "local": "N/A",
    }

    if normalized_name.startswith("oil_"):
        base_info["equipamento"] = "Medidor multifásico"
        base_info["local"] = "Linha de produção / separador"
        base_info["unidade"] = "Sm3/d"
        base_info["descricao"] = f"Indicador derivado de óleo: {normalized_name}"
    elif normalized_name.startswith("gas_"):
        base_info["equipamento"] = "Medidor de gás"
        base_info["local"] = "Linha de gás / separador"
        base_info["unidade"] = "Sm3/d"
        base_info["descricao"] = f"Indicador derivado de gás: {normalized_name}"
    elif normalized_name.startswith("water_"):
        base_info["equipamento"] = "Medidor multifásico"
        base_info["local"] = "Linha de produção / separador"
        base_info["unidade"] = "Sm3/d"
        base_info["descricao"] = f"Indicador derivado de água: {normalized_name}"

    if normalized_name.endswith("_cumulative"):
        base_info["natureza"] = "volume"
        base_info["unidade"] = "Sm3"
    elif normalized_name.endswith("_velocity"):
        base_info["natureza"] = "velocidade"
        base_info["unidade"] = "Sm3/dia"
    elif normalized_name.endswith("_acceleration"):
        base_info["natureza"] = "aceleração"
        base_info["unidade"] = "Sm3/dia²"
    elif "pct_change" in normalized_name or "_roc_" in normalized_name:
        base_info["natureza"] = "variação percentual"
        base_info["unidade"] = "%"
    elif "roll_std" in normalized_name or "expanding_std" in normalized_name:
        base_info["natureza"] = "dispersão"
    elif "volatility" in normalized_name:
        base_info["natureza"] = "volatilidade"
        base_info["unidade"] = "adimensional"
    elif "trend_strength" in normalized_name or normalized_name.endswith("_vs_trend"):
        base_info["natureza"] = "tendência"
        base_info["unidade"] = "adimensional"
    elif "momentum" in normalized_name:
        base_info["natureza"] = "momentum"
    elif "zscore" in normalized_name:
        base_info["natureza"] = "desvio padronizado"
        base_info["unidade"] = "adimensional"
    elif any(
        marker in normalized_name
        for marker in ("lag", "roll_mean", "ewma", "delta", "expanding_mean")
    ):
        base_info["natureza"] = "vazão"

    return base_info

def build_dictionary_context_for_columns(column_names: List[str], title: str) -> str:
    lines = [title]
    for column_name in column_names:
        normalized_name = safe_str(column_name)
        metadata = infer_metadata_from_column_name(normalized_name)
        sql_type = SCHEMA_TYPES_BY_NAME.get(normalized_name, metadata.get("tipo", "TEXT"))
        rendered_line = (
            f"- {normalized_name} | sql_type={sql_type} "
            f"| descricao={metadata['descricao']} "
            f"| unidade={metadata['unidade']} "
            f"| natureza={metadata['natureza']} "
            f"| equipamento={metadata['equipamento']} "
            f"| local={metadata['local']}"
        )
        lines.append(rendered_line)
    return "\n".join(lines)

def build_query_column_context(sql: str, columns: List[str]) -> str:
    # Depois que o SQL roda, esta função explica ao modelo de resposta o que
    # cada coluna retornada representa e se ela veio do schema ou é um alias.
    lines = ["COLUNAS RETORNADAS PELA CONSULTA:"]
    for column_name in columns:
        normalized_name = safe_str(column_name)
        sql_type = SCHEMA_TYPES_BY_NAME.get(normalized_name, "RESULT")
        origem = "schema" if normalized_name in SCHEMA_TYPES_BY_NAME else "alias_ou_expressao"
        metadata = infer_metadata_from_column_name(normalized_name)
        rendered_line = (
            f"- {normalized_name} | sql_type={sql_type} | origem={origem} "
            f"| descricao={metadata['descricao']} "
            f"| unidade={metadata['unidade']} "
            f"| natureza={metadata['natureza']} "
            f"| equipamento={metadata['equipamento']} "
            f"| local={metadata['local']}"
        )
        lines.append(rendered_line)

    return "\n".join(lines)

def infer_sensor_columns_from_question(question: str) -> List[str]:
    normalized_question = safe_str(question).lower()
    sensor_phrase_map = [
        (("pressão de fundo", "pressao de fundo"), ["AVG_DOWNHOLE_PRESSURE"]),
        (("temperatura de fundo", "temperatura downhole"), ["AVG_DOWNHOLE_TEMPERATURE"]),
        (
            (
                "delta de pressão do tubing",
                "delta de pressao do tubing",
                "pressão do tubing",
                "pressao do tubing",
            ),
            ["AVG_DP_TUBING"],
        ),
        (("pressão do anular", "pressao do anular"), ["AVG_ANNULUS_PRESS"]),
        (
            (
                "pressão na cabeça do poço",
                "pressao na cabeca do poco",
                "pressão de cabeça",
                "pressao de cabeca",
            ),
            ["AVG_WHP_P"],
        ),
        (("temperatura na cabeça do poço", "temperatura na cabeca do poco"), ["AVG_WHT_P"]),
        (("horas por dia", "horas em operação", "horas em operacao"), ["ON_STREAM_HRS"]),
    ]

    matched_columns: List[str] = []
    for phrases, columns in sensor_phrase_map:
        if any(phrase in normalized_question for phrase in phrases):
            for column_name in columns:
                if column_name in SCHEMA_TYPES_BY_NAME and column_name not in matched_columns:
                    matched_columns.append(column_name)

    return matched_columns

def infer_all_fluids_from_text(text: str) -> List[str]:
    normalized_text = safe_str(text).lower()
    fluid_terms = {
        "oil": ("óleo", "oleo", "oil"),
        "water": ("água", "agua", "water", "wat"),
        "gas": ("gás", "gas"),
    }

    detected_fluids: List[str] = []
    for fluid_name, terms in fluid_terms.items():
        if any(term in normalized_text for term in terms):
            detected_fluids.append(fluid_name)

    return detected_fluids

def infer_question_intent(question: str) -> str:
    normalized_question = safe_str(question).lower()

    forecast_terms = (
        "previsão", "previsao", "projeção", "projecao", "estimativa", "estimar",
        "próximo dia", "proximo dia", "d+1", "amanhã", "amanha",
    )
    diagnostic_terms = (
        "média móvel", "media movel", "desvio padrão", "desvio padrao", "volatilidade",
        "tendência", "tendencia", "anomalia", "fora do padrão", "fora do padrao",
        "instabilidade", "risco", "avanço de água", "avanco de agua", "z-score", "zscore",
        "momentum", "aceleração", "aceleracao", "variação percentual", "variacao percentual",
        "comportamento típico", "comportamento tipico", "vs_trend", "trend_strength",
        "acumulado", "oscilação", "oscilacao", "estabilização", "estabilizacao", "perdendo força", "perdendo forca",
    )
    aggregation_terms = (
        "média", "media", "máximo", "maximo", "máxima", "maxima", "mínimo", "minimo",
        "maior", "menor", "pico", "top ", "ranking",
    )
    series_terms = (
        "últimos", "ultimos", "registros", "evoluiu", "evoluíram", "evoluiram",
        "série temporal", "serie temporal", "sequência", "sequencia", "histórico", "historico",
    )

    if any(term in normalized_question for term in forecast_terms):
        return "forecast"
    if any(term in normalized_question for term in diagnostic_terms):
        return "derived_diagnostic"
    if any(term in normalized_question for term in aggregation_terms):
        return "historical_aggregation"
    if any(term in normalized_question for term in series_terms):
        return "raw_series"
    return "raw_point"

def render_question_intent_label(intent: str) -> str:
    intent_labels = {
        "raw_point": "leitura pontual crua",
        "historical_aggregation": "agregação histórica",
        "raw_series": "série temporal de valores crus",
        "derived_diagnostic": "diagnóstico com derivados",
        "forecast": "projeção futura",
    }
    return intent_labels.get(intent, intent)

def is_derived_schema_column(column_name: str) -> bool:
    normalized_name = safe_str(column_name).lower()
    derived_markers = (
        "_lag_", "_roll_mean_", "_roll_std_", "_delta_", "_pct_change_", "_ewma_",
        "_expanding_mean", "_expanding_std", "_cumulative", "_velocity", "_acceleration",
        "_trend_strength", "_vs_trend", "_volatility_index", "_momentum_", "_roc_", "_zscore_",
    )
    return any(marker in normalized_name for marker in derived_markers)

def select_derived_columns_for_fluid(question: str, fluid_name: str, intent: str) -> List[str]:
    normalized_question = safe_str(question).lower()
    selected_columns: List[str] = []

    def add_suffixes(*suffixes: str) -> None:
        for suffix in suffixes:
            candidate_column = f"{fluid_name}_{suffix}"
            if candidate_column in SCHEMA_TYPES_BY_NAME and candidate_column not in selected_columns:
                selected_columns.append(candidate_column)

    if intent == "forecast":
        add_suffixes("velocity", "acceleration", "momentum_30d", "roc_30d", "trend_strength", "vs_trend", "delta_1d")

    if any(term in normalized_question for term in ("semana", "7 dias", "7d")):
        add_suffixes("roll_mean_7", "roll_std_7", "delta_7d", "pct_change_7d", "momentum_7d", "roc_7d")
    if any(term in normalized_question for term in ("14 dias", "14d", "duas semanas")):
        add_suffixes("roll_mean_14", "roll_std_14", "pct_change_14d")
    if any(term in normalized_question for term in ("mês", "mes", "30 dias", "30d", "último mês", "ultimo mes")):
        add_suffixes("roll_mean_30", "roll_std_30", "momentum_30d", "roc_30d", "zscore_30")

    if any(
        term in normalized_question
        for term in (
            "média móvel",
            "media movel",
            "comportamento típico",
            "comportamento tipico",
        )
    ):
        if "7" in normalized_question:
            add_suffixes("roll_mean_7")
        if "14" in normalized_question:
            add_suffixes("roll_mean_14")
        add_suffixes("roll_mean_30")

    if any(
        term in normalized_question
        for term in (
            "desvio padrão",
            "desvio padrao",
            "volatilidade",
            "dispersão",
            "dispersao",
        )
    ):
        if "7" in normalized_question:
            add_suffixes("roll_std_7")
        if "14" in normalized_question:
            add_suffixes("roll_std_14")
        add_suffixes("roll_std_30", "volatility_index")

    if any(
        term in normalized_question
        for term in (
            "tendência",
            "tendencia",
            "comportamento típico",
            "comportamento tipico",
            "vs trend",
            "vs_trend",
        )
    ):
        add_suffixes("trend_strength", "vs_trend", "roll_mean_30")

    if fluid_name == "water" and any(
        term in normalized_question
        for term in ("risco", "avanço de água", "avanco de agua")
    ):
        add_suffixes(
            "roll_mean_30",
            "roll_std_30",
            "trend_strength",
            "vs_trend",
            "delta_7d",
            "pct_change_7d",
            "pct_change_14d",
        )

    if any(term in normalized_question for term in ("alta", "queda", "caiu", "subiu", "variação", "variacao", "ritmo")):
        add_suffixes("delta_1d", "delta_7d", "pct_change_1d", "pct_change_7d", "pct_change_14d")

    if any(
        term in normalized_question
        for term in ("anomalia", "fora do padrão", "fora do padrao", "z-score", "zscore")
    ):
        add_suffixes("zscore_30", "roll_std_30", "trend_strength")

    if any(term in normalized_question for term in ("momentum", "força", "forca")):
        add_suffixes("momentum_7d", "momentum_30d")

    if any(term in normalized_question for term in ("roc", "taxa de variação", "taxa de variacao", "percentual")):
        add_suffixes("roc_7d", "roc_30d", "pct_change_1d", "pct_change_7d", "pct_change_14d")

    if any(term in normalized_question for term in ("acum", "acumulado")):
        add_suffixes("cumulative", "expanding_mean", "expanding_std")

    if intent == "derived_diagnostic" and not selected_columns:
        add_suffixes("roll_mean_30", "roll_std_30", "trend_strength", "vs_trend", "delta_1d", "pct_change_1d")

    return selected_columns

def select_relevant_columns_for_question(question: str) -> List[str]:
    normalized_question = safe_str(question).lower()
    intent = infer_question_intent(question)
    selected_columns: List[str] = []

    def add_columns(*column_names: str) -> None:
        for column_name in column_names:
            if column_name in SCHEMA_TYPES_BY_NAME and column_name not in selected_columns:
                selected_columns.append(column_name)

    add_columns("DATEPRD", "NPD_WELL_BORE_NAME")

    explicit_schema_mentions = [
        column_name
        for column_name in SCHEMA_COLUMN_NAMES
        if column_name.lower() in normalized_question
    ]
    for column_name in explicit_schema_mentions:
        add_columns(column_name)

    if any(term in normalized_question for term in ("horas", "hora", "operação", "operacao", "on stream")):
        add_columns("ON_STREAM_HRS")

    sensor_columns = infer_sensor_columns_from_question(question)
    if not sensor_columns and any(term in normalized_question for term in ("pressões", "pressoes", "pressão", "pressao")):
        sensor_columns = [
            column_name
            for column_name in ["AVG_DOWNHOLE_PRESSURE", "AVG_DP_TUBING", "AVG_ANNULUS_PRESS", "AVG_WHP_P"]
            if column_name in SCHEMA_TYPES_BY_NAME
        ]
    if any(term in normalized_question for term in ("temperatura", "temperaturas")):
        for column_name in ["AVG_DOWNHOLE_TEMPERATURE", "AVG_WHT_P"]:
            if column_name in SCHEMA_TYPES_BY_NAME and column_name not in sensor_columns:
                sensor_columns.append(column_name)
    if "choke" in normalized_question:
        for column_name in ["AVG_CHOKE_SIZE_P", "DP_CHOKE_SIZE"]:
            if column_name in SCHEMA_TYPES_BY_NAME and column_name not in sensor_columns:
                sensor_columns.append(column_name)
    for column_name in sensor_columns:
        add_columns(column_name)

    referenced_fluids = infer_all_fluids_from_text(question)
    if not referenced_fluids and any(term in normalized_question for term in ("produção", "producao", "vazão", "vazao", "volume")):
        referenced_fluids = ["oil", "water", "gas"]

    fluid_volume_columns = {
        "oil": "BORE_OIL_VOL",
        "water": "BORE_WAT_VOL",
        "gas": "BORE_GAS_VOL",
    }

    if any(term in normalized_question for term in ("injeção", "injecao", "injeção de água", "injecao de agua")):
        add_columns("BORE_WI_VOL")

    for fluid_name in referenced_fluids:
        base_volume_column = fluid_volume_columns.get(fluid_name, "")
        if base_volume_column:
            add_columns(base_volume_column)
        if intent in {"derived_diagnostic", "forecast"}:
            for column_name in select_derived_columns_for_fluid(question, fluid_name, intent):
                add_columns(column_name)

    if "water" in referenced_fluids and intent in {"derived_diagnostic", "forecast"}:
        add_columns("BORE_OIL_VOL")

    if referenced_fluids or sensor_columns or "ON_STREAM_HRS" in selected_columns or "BORE_WI_VOL" in selected_columns:
        add_columns("FLOW_KIND")

    if intent in {"derived_diagnostic", "forecast"}:
        add_columns("WELL_TYPE")

    if not selected_columns:
        add_columns("DATEPRD", "NPD_WELL_BORE_NAME", "FLOW_KIND")

    return selected_columns[:20]

def build_relevant_dictionary_context(question: str) -> str:
    intent = infer_question_intent(question)
    selected_columns = select_relevant_columns_for_question(question)
    title = (
        "DICIONÁRIO DE DADOS RELEVANTE PARA ESTA PERGUNTA "
        f"(classe_inferida={render_question_intent_label(intent)}):"
    )
    return build_dictionary_context_for_columns(selected_columns, title)

def build_relevant_schema_context(question: str) -> str:
    intent = infer_question_intent(question)
    selected_columns = select_relevant_columns_for_question(question)
    primary_columns = [column_name for column_name in selected_columns if not is_derived_schema_column(column_name)]
    derived_columns = [column_name for column_name in selected_columns if is_derived_schema_column(column_name)]

    context_lines = ["GUIA SEMÂNTICO RELEVANTE DO SCHEMA REAL:"]
    context_lines.append(f"- classe_inferida_da_pergunta: {render_question_intent_label(intent)}")
    if primary_columns:
        context_lines.append(f"- colunas_primarias_prioritarias: {', '.join(primary_columns)}")
    if derived_columns:
        context_lines.append(f"- colunas_derivadas_de_suporte: {', '.join(derived_columns)}")

    return "\n".join(context_lines)

def build_sql_column_selection_policy() -> str:
    return dedent("""
POLÍTICA OBRIGATÓRIA DE ESCOLHA DE COLUNAS E EXPRESSÕES SQL:
1. Antes de escrever o SQL, classifique mentalmente a pergunta em uma destas classes: leitura pontual crua, agregação histórica, série temporal de leituras cruas, diagnóstico com derivados, projeção futura.
2. Se a pergunta pedir leitura, valor registrado, valor no dia, leitura mais recente, último valor, medição de uma data ou evolução dos últimos N registros, trate isso como consulta de valores primários/crus armazenados na linha do tempo.
3. Se a pergunta pedir média, máximo, mínimo, pico, menor valor, média do período ou média por dia, trate isso como agregação histórica sobre colunas primárias/cruas, nunca sobre colunas derivadas, salvo pedido explícito em contrário.
4. Se a pergunta pedir média móvel, desvio padrão móvel, tendência, momentum, aceleração, z-score, razão contra tendência, acumulado, ewma, variação percentual ou variação em janelas, trate isso como uso de colunas derivadas já existentes no schema.
5. Em perguntas de risco, diagnóstico analítico, avanço de água, anomalia, instabilidade ou previsão, ancore sempre a resposta em pelo menos uma coluna primária/crua e use as derivadas apenas como suporte explicativo.
6. Colunas primárias/cruas do schema representam leituras ou volumes da própria linha diária: DATEPRD, ON_STREAM_HRS, AVG_DOWNHOLE_PRESSURE, AVG_DOWNHOLE_TEMPERATURE, AVG_DP_TUBING, AVG_ANNULUS_PRESS, AVG_CHOKE_SIZE_P, AVG_WHP_P, AVG_WHT_P, DP_CHOKE_SIZE, BORE_OIL_VOL, BORE_GAS_VOL, BORE_WAT_VOL, BORE_WI_VOL, FLOW_KIND, WELL_TYPE.
7. O prefixo AVG_ em nomes como AVG_DOWNHOLE_PRESSURE, AVG_WHP_P e AVG_WHT_P faz parte do nome da variável armazenada na tabela. Isso NÃO autoriza aplicar automaticamente a função SQL AVG() nessas colunas.
8. Colunas derivadas do schema são as que contêm padrões como: _lag_, _roll_mean_, _roll_std_, _delta_, _pct_change_, _ewma_, _expanding_mean, _expanding_std, _cumulative, _velocity, _acceleration, _trend_strength, _vs_trend, _volatility_index, _momentum_, _roc_, _zscore_.
9. Para leitura pontual crua de produção de óleo, gás, água ou horas de operação, use BORE_OIL_VOL, BORE_GAS_VOL, BORE_WAT_VOL ou ON_STREAM_HRS na linha da data desejada ou na linha mais recente, sem agregar.
10. Para leitura pontual crua de sensores de pressão, temperatura, choke ou tubing, use as colunas primárias do dia: AVG_DOWNHOLE_PRESSURE, AVG_DOWNHOLE_TEMPERATURE, AVG_DP_TUBING, AVG_ANNULUS_PRESS, AVG_CHOKE_SIZE_P, AVG_WHP_P, AVG_WHT_P, DP_CHOKE_SIZE.
11. Para perguntas de histórico recente como 'últimos 10 registros', 'como evoluiu' ou 'mostrar sequência temporal', retorne DATEPRD mais as colunas primárias/cruas pedidas, ordenando por DATEPRD DESC e usando LIMIT adequado. Não colapse em AVG/MIN/MAX se o usuário não pediu agregação.
12. Para perguntas de média simples ao longo do tempo, use AVG() sobre a coluna primária/crua correta. Exemplo: média de horas -> AVG(ON_STREAM_HRS); média de óleo -> AVG(BORE_OIL_VOL).
13. Para perguntas de máximo ou mínimo histórico, use a coluna primária/crua correta e ordene por ela, ou use agregação equivalente. Não use _roll_mean_, _trend_strength ou outras derivadas no lugar do valor real.
14. Para perguntas sobre média móvel, nunca substitua por AVG() temporal. Use a coluna derivada correspondente, como oil_roll_mean_30, gas_roll_mean_30 ou water_roll_mean_30, ou a janela explicitamente pedida quando existir.
15. Para perguntas sobre desvio padrão/volatilidade recente, prefira colunas derivadas já disponíveis como oil_roll_std_30, water_roll_std_30, gas_roll_std_30, oil_volatility_index. Não invente STDDEV() se o schema já traz a métrica pronta.
16. Para perguntas sobre variação percentual, use _pct_change_1d, _pct_change_7d, _pct_change_14d ou _roc_7d/_roc_30d, conforme o horizonte pedido.
17. Para perguntas sobre variação absoluta em dias, use _delta_1d, _delta_3d ou _delta_7d quando disponíveis.
18. Para perguntas sobre tendência, força relativa à tendência ou desvio do comportamento típico, use _trend_strength, _vs_trend, _roll_mean_30 e _zscore_30 como suporte, mas mantenha junto o valor bruto atual BORE_OIL_VOL, BORE_GAS_VOL ou BORE_WAT_VOL.
19. Para perguntas de avanço de água, combine BORE_WAT_VOL e BORE_OIL_VOL com water_roll_mean_30, water_roll_std_30, water_trend_strength, water_vs_trend, water_delta_7d e water_pct_change_7d/14d quando existirem e fizerem sentido.
20. Para perguntas de previsão, use o valor bruto atual mais derivadas de ritmo e tendência como _momentum_30d, _acceleration, _roc_30d, _trend_strength e _vs_trend. Não invente colunas futuras e não projete além de D+1.
21. Para perguntas que pedem o valor mais recente de uma métrica, priorize ORDER BY DATEPRD DESC LIMIT 1 e, quando a métrica puder estar ausente, filtre a métrica principal com IS NOT NULL.
22. Para perguntas de produção, leituras recentes ou histórico produtivo, priorize FLOW_KIND = 'production'. Para BORE_WI_VOL ou cenários de injeção, use FLOW_KIND compatível com injeção quando necessário.
23. Quando a pergunta citar um poço pelo identificador humano curto, como 15/9-F-5, use NPD_WELL_BORE_NAME. Não use WELL_BORE_CODE para igualdade textual nesses casos.
24. Nunca troque uma coluna primária/crua por uma derivada só porque os nomes parecem semanticamente próximos. Exemplo: leitura atual de água é BORE_WAT_VOL; média móvel de água é water_roll_mean_30; são conceitos diferentes.
25. Nunca use AVG(coluna_primaria) se a pergunta pediu leitura no dia ou valor registrado. Nunca use coluna derivada se a pergunta pediu valor bruto do dia.
26. Nunca agregue novamente uma coluna derivada sem necessidade clara. Exemplo: evite AVG(water_roll_mean_30), MAX(oil_trend_strength) ou MIN(water_pct_change_7d) salvo pedido explícito e plenamente justificado.
27. Quando houver ambiguidade entre valor bruto e estatística derivada, prefira o valor bruto e acrescente derivadas como colunas auxiliares, não como substitutas.
28. Se a pergunta for respondida por um único valor bruto recente, o SQL deve ser curto e direto. Se a pergunta for analítica, retorne apenas as colunas de suporte realmente úteis, sem excesso de colunas irrelevantes.
29. Antes de finalizar o SQL, faça uma checagem mental: a coluna principal escolhida responde literalmente ao que foi pedido? Ela é crua ou derivada? O uso de AVG/MIN/MAX/ORDER BY está coerente com a intenção da pergunta?
30. Se a pergunta mencionar explicitamente média móvel, desvio padrão móvel, acumulado, tendência, anomalia estatística, ritmo, aceleração, z-score ou projeção, aí sim as colunas derivadas são centrais. Caso contrário, assuma que o usuário quer primeiro o valor bruto da série temporal.
""").strip()

def build_sql_few_shot_catalog() -> Dict[str, str]:
    return {
        "raw_day": dedent(f"""
        [EXEMPLO LEITURA NO DIA]
        Pergunta: Como estavam as produções de óleo e de água do poço 15/9-F-5 em 10/06/2014?
        SQL: SELECT DATEPRD, NPD_WELL_BORE_NAME, FLOW_KIND, BORE_OIL_VOL, BORE_WAT_VOL FROM {TABLE_NAME} WHERE NPD_WELL_BORE_NAME = '15/9-F-5' AND date(DATEPRD) = '2014-06-10'
        """).strip(),
        "raw_recent": dedent(f"""
        [EXEMPLO LEITURA CRUA MAIS RECENTE]
        Pergunta: Qual foi a leitura mais recente da produção de água do poço 15/9-F-5?
        SQL: SELECT DATEPRD, NPD_WELL_BORE_NAME, FLOW_KIND, BORE_WAT_VOL FROM {TABLE_NAME} WHERE NPD_WELL_BORE_NAME = '15/9-F-5' AND FLOW_KIND = 'production' AND BORE_WAT_VOL IS NOT NULL ORDER BY DATEPRD DESC LIMIT 1
        """).strip(),
        "sensor_recent": dedent(f"""
        [EXEMPLO LEITURA REGISTRADA DE SENSOR]
        Pergunta: Qual foi a leitura mais recente da pressão de fundo do poço 15/9-F-5?
        SQL: SELECT DATEPRD, NPD_WELL_BORE_NAME, AVG_DOWNHOLE_PRESSURE FROM {TABLE_NAME} WHERE NPD_WELL_BORE_NAME = '15/9-F-5' AND AVG_DOWNHOLE_PRESSURE IS NOT NULL ORDER BY DATEPRD DESC LIMIT 1
        """).strip(),
        "aggregation_mean": dedent(f"""
        [EXEMPLO AGREGAÇÃO HISTÓRICA - MÉDIA]
        Pergunta: Em média, quantas horas por dia o poço 15/9-F-5 ficou em operação?
        SQL: SELECT AVG(ON_STREAM_HRS) AS avg_on_stream_hrs FROM {TABLE_NAME} WHERE NPD_WELL_BORE_NAME = '15/9-F-5' AND FLOW_KIND = 'production' AND ON_STREAM_HRS IS NOT NULL
        """).strip(),
        "aggregation_max": dedent(f"""
        [EXEMPLO AGREGAÇÃO HISTÓRICA - MAIOR VALOR]
        Pergunta: Em que dia o poço 15/9-F-5 atingiu sua maior produção de óleo e qual foi esse volume?
        SQL: SELECT DATEPRD, NPD_WELL_BORE_NAME, FLOW_KIND, BORE_OIL_VOL FROM {TABLE_NAME} WHERE NPD_WELL_BORE_NAME = '15/9-F-5' AND FLOW_KIND = 'production' AND BORE_OIL_VOL IS NOT NULL ORDER BY BORE_OIL_VOL DESC LIMIT 1
        """).strip(),
        "aggregation_top_n": dedent(f"""
        [EXEMPLO RANKING HISTÓRICO]
        Pergunta: Quais foram os 5 dias com maior produção de água do 15/9-F-5?
        SQL: SELECT DATEPRD, NPD_WELL_BORE_NAME, FLOW_KIND, BORE_WAT_VOL FROM {TABLE_NAME} WHERE NPD_WELL_BORE_NAME = '15/9-F-5' AND FLOW_KIND = 'production' AND BORE_WAT_VOL IS NOT NULL ORDER BY BORE_WAT_VOL DESC LIMIT 5
        """).strip(),
        "raw_series": dedent(f"""
        [EXEMPLO SÉRIE TEMPORAL DE VALORES CRUDOS]
        Pergunta: Nos 10 registros mais recentes, como evoluíram a data, a produção de óleo e a produção de água do poço 15/9-F-5?
        SQL: SELECT DATEPRD, NPD_WELL_BORE_NAME, FLOW_KIND, BORE_OIL_VOL, BORE_WAT_VOL FROM {TABLE_NAME} WHERE NPD_WELL_BORE_NAME = '15/9-F-5' AND FLOW_KIND = 'production' ORDER BY DATEPRD DESC LIMIT 10
        """).strip(),
        "derived_window": dedent(f"""
        [EXEMPLO MÉTRICA DERIVADA EXPLÍCITA]
        Pergunta: Qual é a média móvel de 30 períodos da água no registro mais recente do poço 15/9-F-5?
        SQL: SELECT DATEPRD, NPD_WELL_BORE_NAME, FLOW_KIND, BORE_WAT_VOL, water_roll_mean_30 FROM {TABLE_NAME} WHERE NPD_WELL_BORE_NAME = '15/9-F-5' AND FLOW_KIND = 'production' AND water_roll_mean_30 IS NOT NULL ORDER BY DATEPRD DESC LIMIT 1
        """).strip(),
        "diagnostic_water": dedent(f"""
        [EXEMPLO DIAGNÓSTICO DE ÁGUA]
        Pergunta: O dado mais recente sugere aumento de risco de avanço de água? Mostre os sinais que sustentam essa leitura.
        SQL: SELECT DATEPRD, NPD_WELL_BORE_NAME, FLOW_KIND, BORE_WAT_VOL, water_roll_mean_30, water_roll_std_30, water_trend_strength, water_vs_trend, water_delta_7d, BORE_OIL_VOL FROM {TABLE_NAME} WHERE FLOW_KIND = 'production' ORDER BY DATEPRD DESC LIMIT 1
        """).strip(),
        "anomaly_oil": dedent(f"""
        [EXEMPLO ANOMALIA ESTATÍSTICA]
        Pergunta: Em que datas tivemos anomalias graves de produção de óleo fora do padrão estatístico dos últimos 30 dias?
        SQL: SELECT DATEPRD, NPD_WELL_BORE_NAME, BORE_OIL_VOL, oil_zscore_30 FROM {TABLE_NAME} WHERE FLOW_KIND = 'production' AND abs(oil_zscore_30) > 2.0 ORDER BY DATEPRD DESC
        """).strip(),
        "forecast_oil": dedent(f"""
        [EXEMPLO PROJEÇÃO CURTÍSSIMO PRAZO]
        Pergunta: Pelo ritmo recente de produção do poço 15/9-F-5 e pela aceleração observada, qual seria a estimativa de produção de óleo para o próximo dia?
        SQL: SELECT DATEPRD, NPD_WELL_BORE_NAME, FLOW_KIND, BORE_OIL_VOL, oil_velocity, oil_acceleration, oil_momentum_30d, oil_roc_30d, MAX(0, BORE_OIL_VOL + oil_velocity) AS proj_oleo_proximo_dia FROM {TABLE_NAME} WHERE NPD_WELL_BORE_NAME = '15/9-F-5' AND FLOW_KIND = 'production' AND BORE_OIL_VOL IS NOT NULL ORDER BY DATEPRD DESC LIMIT 1
        """).strip(),
    }

def select_relevant_few_shots(question: str) -> str:
    normalized_question = safe_str(question).lower()
    intent = infer_question_intent(question)
    referenced_fluids = infer_all_fluids_from_text(question)
    sensor_columns = infer_sensor_columns_from_question(question)
    catalog = build_sql_few_shot_catalog()
    selected_keys: List[str] = []

    def add_key(key: str) -> None:
        if key in catalog and key not in selected_keys:
            selected_keys.append(key)

    if sensor_columns:
        add_key("sensor_recent")

    if intent == "historical_aggregation":
        if any(term in normalized_question for term in ("média", "media")):
            add_key("aggregation_mean")
        if any(term in normalized_question for term in ("maior", "máximo", "maximo", "pico", "top ", "ranking")):
            add_key("aggregation_top_n")
            add_key("aggregation_max")
        add_key("raw_day")
    elif intent == "raw_series":
        add_key("raw_series")
        add_key("raw_recent")
    elif intent == "derived_diagnostic":
        add_key("derived_window")
        if "water" in referenced_fluids or any(term in normalized_question for term in ("água", "agua", "water", "avanço de água", "avanco de agua")):
            add_key("diagnostic_water")
        add_key("anomaly_oil")
        add_key("raw_recent")
    elif intent == "forecast":
        add_key("forecast_oil")
        add_key("derived_window")
        add_key("raw_recent")
    else:
        if any(term in normalized_question for term in ("mais recente", "último", "ultimo", "registrada")):
            add_key("raw_recent")
        if any(term in normalized_question for term in ("data", "dia", "em ")):
            add_key("raw_day")

    if not selected_keys:
        add_key("raw_day")
        add_key("raw_recent")

    rendered_examples = "\n\n".join(catalog[key] for key in selected_keys)
    return f"=== EXEMPLOS RELEVANTES DE TRADUÇÃO PARA SQLITE ===\n\n{rendered_examples}".strip()

# -----------------------------------------------------------------------------
# 1. CONFIGURAÇÃO DE SEGURANÇA E CONEXÃO OPENROUTER
# -----------------------------------------------------------------------------
# Esta é a variável obrigatória para a execução remota do notebook.
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY", "SUA_CHAVE_OPENROUTER_AQUI")

# PIPELINE_AUTO_RUN controla se o notebook executa o fluxo automaticamente ao
# rodar a célula. Em estudo, isso é útil para alternar entre:
# - modo automático
# - modo exploratório/manual
# O nome NOTEBOOK_10_1_AUTO_RUN é o preferencial deste notebook.
# NOTEBOOK_09_AUTO_RUN fica como fallback para compatibilidade com versões
# anteriores do material.
PIPELINE_AUTO_RUN = os.getenv(
    "NOTEBOOK_10_1_AUTO_RUN",
    os.getenv("NOTEBOOK_09_AUTO_RUN", "1"),
).strip() != "0"

# Banco interno de perguntas de teste.
# Repare que aqui a linguagem é humana, não linguagem de banco.
PIPELINE_QUESTION_BANK = [
    {
        "categoria": "simples",
        "pergunta": "Como estavam as produções de óleo e de água do poço 15/9-F-5 em 10/06/2014?",
    },
    {
        "categoria": "simples",
        "pergunta": "Qual foi a leitura mais recente da pressão de fundo do poço 15/9-F-5?",
    },
    {
        "categoria": "simples",
        "pergunta": "Em média, quantas horas por dia o poço 15/9-F-5 ficou em operação?",
    },
    {
        "categoria": "normal",
        "pergunta": "Em que dia o poço 15/9-F-5 atingiu sua maior produção de óleo e qual foi esse volume?",
    },
    {
        "categoria": "normal",
        "pergunta": "Quais foram os 5 dias com maior produção de água do 15/9-F-5?",
    },
    {
        "categoria": "normal",
        "pergunta": "Nos 10 registros mais recentes, como evoluíram a data, a produção de óleo e a produção de água do poço 15/9-F-5?",
    },
    {
        "categoria": "complexa",
        "pergunta": "A produção de óleo do poço 15/9-F-5 mais recente ficou acima ou abaixo do comportamento típico do último mês? Mostre também se houve alta ou queda no dia e se o valor parece fora do padrão.",
    },
    {
        "categoria": "complexa",
        "pergunta": "Em que datas a produção de óleo do poço 15/9-F-5 ficou claramente fora do padrão normal do último mês?",
    },
    {
        "categoria": "complexa",
        "pergunta": "O dado mais recente sugere aumento de risco de avanço de água no poço 15/9-F-5? Mostre os sinais que sustentam essa conclusão.",
    },
    {
        "categoria": "complexa",
        "pergunta": "Pelo ritmo recente de produção do poço 15/9-F-5 e pela aceleração observada, qual seria a estimativa de produção de óleo para o próximo dia e quais sinais apoiam essa leitura?",
    },
    {
        "categoria": "complexa",
        "pergunta": "Observando o histórico acumulado e a oscilação da produção, o poço 15/9-F-5 está perdendo força de forma contínua ou entrando em estabilização?",
    },
    {
        "categoria": "complexa",
        "pergunta": "Com base nas pressões mais recentes e no comportamento da produção, existe sinal de instabilidade mecânica de curtíssimo prazo no poço 15/9-F-5?",
    },
]

def choose_pipeline_question() -> Dict[str, Any]:
    # Sorteia uma pergunta e também devolve o índice, o que ajuda em logs,
    # reprodutibilidade e depuração de casos específicos.
    selected_index = int(np.random.choice(len(PIPELINE_QUESTION_BANK)))
    selected_entry = dict(PIPELINE_QUESTION_BANK[selected_index])
    selected_entry["indice"] = selected_index
    return selected_entry

# =============================================================================
# NOVA CAMADA DE ESTRUTURAÇÃO DETERMINÍSTICA (PYDANTIC AI)
# =============================================================================

class SqlGenerationOutput(BaseModel):
    """Schema estrito para forçar o LLM a entregar dados tipados e limpos."""
    query_sql: str = Field(
        description="Código SQL puro e limpo pronto para execução direta no SQLite. Sem blocos markdown (```sql)."
    )
    justificativa_analitica: str = Field(
        description="Breve explicação das colunas analíticas e temporais do Volve escolhidas para o SQL."
    )

class OperatorResponseSchema(BaseModel):
    texto_resposta: str = Field(
        description="Resposta direta, clara e em português para o operador. Use Markdown."
    )
    alerta_critico: bool = Field(
        description="Defina como True apenas se os dados indicarem anomalias severas ou quebra de limites."
    )

# Configuração do provider OpenAI-compatible para o OpenRouter
pydantic_ai_provider = OpenAIProvider(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
)

# Configuração do modelo encapsulado no padrão PydanticAI
pydantic_ai_model = OpenAIChatModel(
    model_name=REMOTE_SQL_MODEL,
    provider=pydantic_ai_provider,
)

# Agente PydanticAI especialista em Text-to-SQL estruturado
sql_extractor_agent = Agent(
    model=pydantic_ai_model,
    output_type=SqlGenerationOutput,
    system_prompt=(
        "Você é um engenheiro sênior de dados especialista em SQLite para Oil & Gas.\n"
        "Sua saída deve obedecer estritamente ao schema de dados JSON estruturado requisitado.\n"
        "Antes de gerar o SQL, classifique a pergunta entre leitura crua, agregação histórica, série temporal, diagnóstico com derivados ou projeção.\n"
        "Use colunas primárias/cruas para leituras do dia, valores registrados, históricos recentes e agregações simples.\n"
        "Use colunas derivadas apenas quando a pergunta pedir explicitamente estatística, tendência, anomalia, variação, risco ou previsão.\n"
        "Nunca confunda o prefixo AVG_ no nome de uma coluna com a necessidade de aplicar a função SQL AVG()."
    )
)

pydantic_ai_response_model = OpenAIChatModel(
    model_name=REMOTE_TEXT_MODEL,
    provider=pydantic_ai_provider,
)

response_synthesizer_agent = Agent(
    model=pydantic_ai_response_model,
    output_type=OperatorResponseSchema,
    system_prompt=(
        "Você é um Assistente de Engenharia de Operações Especialista no Campo de Volve.\n"
        "Sua tarefa é traduzir tabelas de dados brutos em insights diretos para operadores de campo.\n"
        "Diretrizes:\n"
        "1. Seja direto. Evite introduções longas como 'Com base nos dados...'. Vá direto ao ponto.\n"
        "2. Sempre inclua as unidades corretas (bar, °C, Sm3/d, %).\n"
        "3. Use tom industrial direto, com foco em segurança operacional e verificações objetivas.\n"
        "4. Se a tabela enviada estiver vazia, explique que não há registros no intervalo solicitado.\n"
        "5. Marque alerta_critico como True apenas se os dados indicarem anomalias severas, quebra de limites ou risco operacional claro."
    )
)
# Palavras-chave proibidas estritas para o Guardrail de Escrita
# Mesmo que o prompt peça "somente SELECT", criamos uma defesa adicional.
PROHIBITED_KEYWORDS = {"DROP", "DELETE", "INSERT", "UPDATE", "ALTER", "CREATE", "TRUNCATE", "EXECUTE", "REPLACE"}

# -----------------------------------------------------------------------------
# 2. CAMADA RIGOROSA DE GUARDRAIL E VALIDAÇÃO ESTÁTICA
# -----------------------------------------------------------------------------
def detect_implausible_forecast_sql(sql_query: str) -> str:
    # Este guardrail foi criado porque o LLM pode tentar extrapolar demais.
    # Como este fluxo é SQL + contexto analítico, aceitamos apenas horizonte D+1
    # para projeções diretas.
    normalized_sql = " ".join(safe_str(sql_query).lower().split())

    # Bloqueia aliases como proj_oleo_7_dias, proj_oleo_30_dias etc.
    forecast_alias_matches = re.findall(r"\bproj_[a-z0-9_]*?(\d+)_dias\b", normalized_sql)
    for horizon_text in forecast_alias_matches:
        if int(horizon_text) > 1:
            return (
                "Guardrail preditivo: projecoes SQL para horizontes acima de D+1 foram bloqueadas. "
                "Retorne apenas D+1 e os indicadores tecnicos de suporte."
            )

    # Bloqueia uma forma específica de extrapolação quadrática que tende a
    # parecer "inteligente", mas pode ser operacionalmente irresponsável.
    if re.search(r"0\.5\s*\*\s*oil_acceleration\s*\*\s*\(\s*([2-9]\d*)\s*\*\s*\1\s*\)", normalized_sql):
        return (
            "Guardrail preditivo: extrapolacao quadratica com aceleracao para horizontes maiores que D+1 foi bloqueada. "
            "Use apenas projecao de curtissimo prazo ou retorne indicadores para analise narrativa."
        )

    return ""

def validar_sql_seguro_e_compativel(sql_query: str) -> tuple[bool, str]:
    """Retorna (True, "") se o SQL for seguro e compatível com SQLite."""

    # Primeiro validamos segurança lexical.
    # Não basta "confiar" no modelo.
    sql_limpo = sql_query.strip().upper()
    
    if any(keyword in sql_limpo for keyword in PROHIBITED_KEYWORDS):
        return False, "Bloqueio de Segurança: Comando de modificação/escrita detectado!"

    # Depois validamos plausibilidade de forecast para este caso de uso.
    forecast_guardrail_message = detect_implausible_forecast_sql(sql_query)
    if forecast_guardrail_message:
        return False, forecast_guardrail_message
        
    try:
        # sqlglot funciona aqui como um parser estático.
        # Ele ajuda a pegar erro de sintaxe antes de bater no banco.
        sqlglot.parse_one(sql_query, read="sqlite")
        return True, ""
    except sqlglot.errors.ParseError as e:
        return False, f"Erro de sintaxe estática (Dialeto SQLite): {str(e)}"

# -----------------------------------------------------------------------------
# 3. PROMPT DE ENGENHARIA DE PROMPT COM FEW-SHOT (MECÂNICA E PREVISÕES)
# -----------------------------------------------------------------------------
def build_sql_prompt(question: str, error_message: str) -> str:
    # Quando a primeira tentativa falha, o erro do SQLite volta para o modelo.
    # Isso cria um loop de autocorreção orientado por evidência real.
    retry_context = ""
    if error_message:
        retry_context = (
            "\nATENÇÃO: sua tentativa anterior falhou com o erro: "
            f"{error_message}. Corrija a sintaxe para o SQLite."
        )

    enriched_question = enrich_question_with_schema_hints(question)
    relevant_schema_context = build_relevant_schema_context(question)
    relevant_dictionary_context = build_relevant_dictionary_context(question)
    column_selection_policy = build_sql_column_selection_policy()
    few_shot_examples = select_relevant_few_shots(question)

    # Ordem do prompt:
    # 1. schema completo
    # 2. contexto semântico dinâmico
    # 3. política obrigatória
    # 4. exemplos relevantes
    # 5. dicionário relevante da pergunta
    # 6. contexto de retry e pergunta
    prompt = f"""
Esquema:
{schema_text}

{relevant_schema_context}

{column_selection_policy}

{few_shot_examples}

{relevant_dictionary_context}

Sua tarefa é gerar SQL SQLite para uma série temporal real de produção offshore do projeto Volve, no Mar do Norte da Noruega.
Tabela disponível: {TABLE_NAME}

Regras Cruciais:
1. Retorne somente SQL puro, sem markdown ou blocos de código.
2. Use apenas comandos SELECT de leitura.
3. Use exatamente os nomes das colunas disponíveis no esquema.
4. A coluna DATEPRD é TIMESTAMP. Quando a pergunta vier com data calendário sem hora, compare usando date(DATEPRD) = 'YYYY-MM-DD' ou date(DATEPRD) em filtros equivalentes; não use igualdade direta DATEPRD = 'YYYY-MM-DD'.
5. Quando a pergunta citar nomes curtos de poço como 15/9-F-5, 15/9-F-4, 15/9-F-15 D ou equivalentes, filtre pela coluna NPD_WELL_BORE_NAME. WELL_BORE_CODE contém rótulos expandidos como 'NO 15/9-F-5 AH' e só deve ser usado se esse formato completo aparecer explicitamente.
6. Quando a pergunta tratar de leitura mais recente, tendência, previsão, maiores volumes ou histórico produtivo de óleo, água ou gás, priorize FLOW_KIND = 'production' e evite linhas em que a métrica principal esteja nula. Em perguntas de data específica, você pode consultar a data informada sem esse filtro e retornar FLOW_KIND para explicar períodos de injeção ou ausência de produção.
7. Para perguntas de risco de avanço de água, use os campos reais disponíveis para água: BORE_WAT_VOL, water_roll_mean_30, water_roll_std_30, water_trend_strength, water_vs_trend, water_delta_7d e water_pct_change_7d/14d quando não estiverem nulos.
8. Não invente colunas como WATER_TOTAL_VOL, OIL_TOTAL_VOL, GAS_TOTAL_VOL ou water_zscore_30. Use apenas os nomes do schema real.
9. Para análises de previsão ou tendências futuras, utilize as colunas matemáticas derivadas disponíveis no ecossistema (_trend_strength, _acceleration, _momentum_30d, _roc_30d, _vs_trend) para sustentar o relatório técnico.
10. Quando a pergunta pedir projeção sem horizonte explicitamente definido, retorne somente projeção de curtíssimo prazo D+1.
11. Não gere colunas de projeção multi-dia como proj_oleo_3_dias, proj_oleo_7_dias ou equivalentes.
12. Não use extrapolação quadrática com aceleração para horizontes acima de D+1.
13. Nunca deixe uma projeção volumétrica ficar negativa; quando necessário, aplique piso físico em zero com MAX(0, ...).
14. Para horizontes maiores que D+1, retorne os indicadores de suporte (momentum, aceleração, roc, trend_strength, zscore) e deixe a interpretação para o parecer técnico final.
15. Quando a pergunta pedir leitura no dia, leitura registrada, último valor ou medição mais recente, retorne a coluna primária correspondente na linha do dia; não use AVG(coluna) entre dias, a menos que o usuário peça média explicitamente.
16. O prefixo técnico AVG_ no nome de uma coluna do schema faz parte do nome da variável armazenada na tabela; isso não significa que você deve aplicar a função SQL AVG() nessa coluna.

Contexto Auxiliar Selecionado:
{retry_context}

Pergunta: {enriched_question}
SQL:
"""
    final_prompt = dedent(prompt).strip()

    return final_prompt

# -----------------------------------------------------------------------------
# 4. NÓ DE GERAÇÃO SQL
# -----------------------------------------------------------------------------
# Este nó decide entre:
# - usar um atalho heurístico (fast path)
# - ou chamar o modelo remoto para gerar SQL
# -----------------------------------------------------------------------------
async def generate_sql_node(state: AgentState) -> Dict[str, Any]:
    question = safe_str(state["question"])
    retry_count = state.get("retry_count", 0)
    attempt_number = retry_count + 1
    log_progress(f"[PERGUNTA]\n{question}")

    # Primeiro tentamos o caminho barato e determinístico.
    # Se funcionar, evitamos custo de LLM.
    fast_path_sql = try_build_rule_based_sql(question)
    if fast_path_sql:
        normalized_fast_path_sql, schema_error = normalize_and_repair_sql(
            fast_path_sql
        )
        log_progress("[PROMPT_SQL_TAMANHO]\nchars=0")
        log_progress("[PROMPT_SQL]\n[nao_utilizado - fast path heuristico]")
        if schema_error:
            return {
                "generated_sql": "",
                "retry_count": retry_count + 1,
                "error_message": schema_error,
                "sql_generation_time": state.get("sql_generation_time", 0.0),
                "local_prompt_chars": state.get("local_prompt_chars", 0),
            }
        log_progress(f"[SQL][GERACAO]\n{safe_str(normalized_fast_path_sql)}")
        return {
            "generated_sql": safe_str(normalized_fast_path_sql),
            "retry_count": retry_count + 1,
            "error_message": "",
            "sql_generation_time": state.get("sql_generation_time", 0.0),
            "local_prompt_chars": state.get("local_prompt_chars", 0),
        }

    error_message = safe_str(state.get("error_message", ""))
    prompt_corpo = build_sql_prompt(question, error_message)

    log_progress(f"[PROMPT_SQL_TAMANHO]\nchars={len(prompt_corpo)}")
    log_progress(f"[PROMPT_SQL]\n{build_sql_prompt_for_display(prompt_corpo)}")
    sql_generation_start_time = time.time()

    try:
        model_settings = {}
        # Alguns modelos Gemini aceitam orçamento de raciocínio via extra_body.
        # Em outros modelos deixamos vazio para evitar parâmetros inválidos.
        if "gemini-2.5" in REMOTE_SQL_MODEL.lower():
            model_settings = {
                "extra_body": {
                    "thinking_config": {
                        "thinking_budget": 1024
                    }
                }
            }

        result = await sql_extractor_agent.run(
            user_prompt=prompt_corpo,
            model_settings=model_settings,
        )

        clean_sql, schema_error = normalize_and_repair_sql(
            result.output.query_sql
        )

        if schema_error:
            elapsed = time.time() - sql_generation_start_time
            return build_sql_generation_error(state, schema_error, elapsed, len(prompt_corpo))

        elapsed = time.time() - sql_generation_start_time
        prompt_chars_estimado = len(prompt_corpo)
        log_progress(f"[SQL][GERACAO]\n{clean_sql}")

        return {
            "generated_sql": clean_sql,
            "retry_count": attempt_number,
            "error_message": "",
            "sql_generation_time": state.get("sql_generation_time", 0.0) + elapsed,
            "local_prompt_chars": state.get("local_prompt_chars", 0) + prompt_chars_estimado,
        }

    except Exception as exc:
        elapsed = time.time() - sql_generation_start_time
        detail = safe_str(str(exc))
        log_progress(f"[SQL][GERACAO][ERRO]\n{detail}")
        return build_sql_generation_error(state, f"Erro na camada PydanticAI: {detail}", elapsed, 0)

# -----------------------------------------------------------------------------
# 5. NÓ DE EXECUÇÃO SQL
# -----------------------------------------------------------------------------
# Aqui o SQL já foi gerado. Este nó:
# - bloqueia query insegura
# - executa no SQLite
# - prepara o resultado para o modelo final
# -----------------------------------------------------------------------------
def execute_sql_node(state: AgentState) -> Dict[str, Any]:
    error_message = safe_str(state.get("error_message", ""))

    # Se o erro veio da API remota, não faz sentido tentar rodar SQL vazio.
    remote_error_markers = (
        "Erro na API OpenRouter",
        "Erro na camada PydanticAI",
    )
    if any(marker in error_message for marker in remote_error_markers):
        return {"db_data": [], "query_result": "", "query_column_context": ""}
        
    generated_sql = safe_str(state.get("generated_sql", "")).strip()
    if not generated_sql:
        return build_empty_execution_response("Nenhum SQL foi gerado pelo modelo.")
        
    # Primeiro rodamos os guardrails locais.
    eh_valido_e_seguro, motivo_erro = validar_sql_seguro_e_compativel(generated_sql)
    if not eh_valido_e_seguro:
        log_progress(f"[SQL][VALIDACAO]\nStatus: BLOQUEADO\nMotivo: {motivo_erro}\nSQL:\n{generated_sql}")
        return build_empty_execution_response(motivo_erro)

    log_progress(f"[SQL][VALIDACAO]\nStatus: OK\nSQL:\n{generated_sql}")
    sql_execution_start_time = time.time()
    try:
        # pd.read_sql é a ponte simples entre SQLite e DataFrame.
        df = pd.read_sql(generated_sql, conn)

        db_data = json.loads(df.to_json(orient="records", date_format="iso"))

        # Guardamos a lista de colunas para explicar semanticamente o resultado
        # ao modelo de resposta.
        query_columns = [safe_str(str(column)) for column in df.columns.tolist()]

        # O resultado é formatado em modo humano antes de ser enviado ao LLM.
        query_result = format_dataframe_for_prompt(df)
        query_column_context = safe_str(build_query_column_context(generated_sql, query_columns))
        elapsed = time.time() - sql_execution_start_time
        log_progress(f"[SQL][EXECUCAO]\n{query_result}")
        return {
            "db_data": db_data,
            "query_result": query_result,
            "query_column_context": query_column_context,
            "error_message": "",
            "sql_execution_time": state.get("sql_execution_time", 0.0) + elapsed,
        }
    except Exception as exc:
        # O próprio erro real do SQLite vira insumo para retry no nó SQL.
        elapsed = time.time() - sql_execution_start_time
        detail = safe_str(str(exc))
        log_progress(f"[SQL][EXECUCAO][ERRO]\n{detail}")
        return {
            "error_message": f"Erro de execução no SQLite: {detail}",
            "db_data": [],
            "query_result": "",
            "query_column_context": "",
            "sql_execution_time": state.get("sql_execution_time", 0.0) + elapsed,
        }

# -----------------------------------------------------------------------------
# 6. NÓ DE RESPOSTA FINAL
# -----------------------------------------------------------------------------
# Este nó pega:
# - a pergunta original
# - o resultado tabular
# - a semântica das colunas
# e transforma isso em uma resposta curta para o operador.
# -----------------------------------------------------------------------------
async def respond_node(state: AgentState) -> Dict[str, Any]:
    question = safe_str(state.get("question", ""))
    generated_sql = safe_str(state.get("generated_sql", ""))
    error_message = safe_str(state.get("error_message", ""))
    db_data = state.get("db_data") or []
    query_column_context = safe_str(state.get("query_column_context", ""))

    if error_message and not db_data:
        failure_text = f"Operador, falha no processamento de dados: {error_message}"
        log_progress(f"[RESPOSTA_MODELO]\n{failure_text}")
        return {
            "final_answer": failure_text,
            "final_response": failure_text,
            "alerta_critico": False,
        }

    if not db_data:
        empty_text = "Pesquisa concluída, porém a base analítica não retornou registros correspondentes."
        log_progress(f"[RESPOSTA_MODELO]\n{empty_text}")
        return {
            "final_answer": empty_text,
            "final_response": empty_text,
            "alerta_critico": False,
        }

    response_start_time = time.time()

    corpo_contexto = (
        f"Pergunta do Operador: {question}\n\n"
        f"SQL Gerado para Consulta: {generated_sql}\n\n"
        f"Dicionário Semântico das Colunas Retornadas:\n"
        f"{query_column_context if query_column_context else 'N/A'}\n\n"
        f"Dados Brutos Retornados do Banco de Dados:\n"
        f"{str(db_data) if db_data else '[]'}\n"
    )

    if error_message:
        corpo_contexto += (
            f"\nNota Técnica: O sistema encontrou o seguinte erro técnico, caso relevante: {error_message}"
        )

    log_progress(f"[PROMPT_RESPOSTA_TAMANHO]\nchars={len(corpo_contexto)}")
    log_progress(f"[PROMPT_RESPOSTA]\n{corpo_contexto}")

    try:
        result = await response_synthesizer_agent.run(
            user_prompt=corpo_contexto,
        )

        resposta_final: OperatorResponseSchema = result.output
        texto_exibicao = safe_str(resposta_final.texto_resposta).strip()
        if resposta_final.alerta_critico:
            texto_exibicao = "⚠️ **[ALERTA DE SEGURANÇA OPERACIONAL]**\n\n" + texto_exibicao

        elapsed = time.time() - response_start_time
        log_progress(f"[RESPOSTA_MODELO]\n{texto_exibicao}")
        return {
            "final_answer": texto_exibicao,
            "final_response": texto_exibicao,
            "alerta_critico": bool(resposta_final.alerta_critico),
            "response_generation_time": state.get("response_generation_time", 0.0) + elapsed,
            "remote_response_time": state.get("remote_response_time", 0.0) + elapsed,
            "remote_prompt_chars": state.get("remote_prompt_chars", 0) + len(corpo_contexto),
        }

    except Exception as exc:
        elapsed = time.time() - response_start_time
        detail = safe_str(str(exc))
        fallback_error_text = (
            "Não foi possível formatar os dados de produção de forma clara neste momento. "
            f"Dados brutos: {str(db_data)}"
        )
        log_progress(f"[RESPOSTA_MODELO][ERRO]\n{detail}")
        return {
            "final_answer": fallback_error_text,
            "final_response": fallback_error_text,
            "alerta_critico": False,
            "error_message": f"Falha no nó de resposta: {detail}",
            "response_generation_time": state.get("response_generation_time", 0.0) + elapsed,
            "remote_response_time": state.get("remote_response_time", 0.0) + elapsed,
        }

# -----------------------------------------------------------------------------
# 7. ORQUESTRAÇÃO DO PIPELINE LANGGRAPH
# -----------------------------------------------------------------------------
def should_retry_or_respond(state: AgentState) -> str:
    # Esta função é um roteador.
    # Se ainda há erro e ainda há tentativas disponíveis, volta ao nó SQL.
    # Caso contrário, segue para a resposta final.
    has_error = bool(state.get("error_message"))
    retry_count = state.get("retry_count", 0)

    if has_error and retry_count < MAX_SQL_RETRIES:
        return "generate_sql"

    return "respond"

def build_workflow_app():
    # O LangGraph é montado explicitamente:
    # generate_sql -> execute_sql -> (retry ou respond)
    workflow = StateGraph(AgentState)
    workflow.add_node("generate_sql", generate_sql_node)
    workflow.add_node("execute_sql", execute_sql_node)
    workflow.add_node("respond", respond_node)
    workflow.add_edge(START, "generate_sql")
    workflow.add_edge("generate_sql", "execute_sql")
    workflow.add_conditional_edges(
        "execute_sql",
        should_retry_or_respond,
        {"generate_sql": "generate_sql", "respond": "respond"},
    )
    workflow.add_edge("respond", END)
    return workflow.compile()

def build_initial_state(question: str) -> AgentState:
    # Estado inicial limpo para cada rodada do pipeline.
    return {
        "question": safe_str(question),
        "generated_sql": "",
        "error_message": "",
        "retry_count": 0,
        "db_data": [],
        "query_result": "",
        "query_column_context": "",
        "sql_generation_time": 0.0,
        "sql_execution_time": 0.0,
        "response_generation_time": 0.0,
        "remote_response_time": 0.0,
        "local_prompt_chars": 0,
        "remote_prompt_chars": 0,
        "final_answer": "",
        "final_response": "",
        "alerta_critico": False,
    }

def validate_remote_configuration() -> None:
    # Valida se a chave da API foi realmente configurada.
    # Isso evita rodar metade do notebook antes de descobrir que a chave falta.
    cleaned_key = safe_str(OPENROUTER_API_KEY).strip()
    if not cleaned_key or cleaned_key == "SUA_CHAVE_OPENROUTER_AQUI":
        raise ValueError(
            "OPENROUTER_API_KEY nao configurada. Defina a chave antes de executar o pipeline remoto."
        )

async def run_pipeline(question: str) -> AgentState:
    # Função de alto nível que executa o grafo inteiro para uma pergunta.
    initial_state = build_initial_state(question)

    # recursion_limit evita loop infinito caso algo saia do previsto.
    return await app.ainvoke(initial_state, {"recursion_limit": 15})

# Compilamos o app uma única vez ao carregar a célula.
app = build_workflow_app()

PIPELINE_OUTPUT = None
PIPELINE_SELECTED_QUESTION = None
if PIPELINE_AUTO_RUN:
    # Modo de execução automática:
    # 1. valida chave
    # 2. sorteia pergunta
    # 3. executa pipeline
    # 4. mostra SQL e resposta final
    validate_remote_configuration()
    PIPELINE_SELECTED_QUESTION = choose_pipeline_question()
    selected_question = safe_str(PIPELINE_SELECTED_QUESTION["pergunta"])
    PIPELINE_OUTPUT = await run_pipeline(selected_question)
else:
    pass


[PERGUNTA]
Em que dia o poço 15/9-F-5 atingiu sua maior produção de óleo e qual foi esse volume?
[PROMPT_SQL_TAMANHO]
chars=14774
[PROMPT_SQL]
Esquema:
DATEPRD (TIMESTAMP), NPD_WELL_BORE_CODE (INTEGER), NPD_WELL_BORE_NAME (TEXT), NPD_FIELD_CODE (INTEGER), NPD_FIELD_NAME (TEXT), NPD_FACILITY_CODE (INTEGER), NPD_FACILITY_NAME (TEXT), ON_STREAM_HRS (REAL), AVG_DOWNHOLE_PRESSURE (REAL), AVG_DOWNHOLE_TEMPERATURE (REAL), AVG_DP_TUBING (REAL), AVG_ANNULUS_PRESS (REAL), AVG_CHOKE_SIZE_P (REAL), AVG_CHOKE_UOM (TEXT), AVG_WHP_P (REAL), AVG_WHT_P (REAL), DP_CHOKE_SIZE (REAL), BORE_OIL_VOL (REAL), BORE_GAS_VOL (REAL), BORE_WAT_VOL (REAL), BORE_WI_VOL (REAL), FLOW_KIND (TEXT), WELL_TYPE (TEXT), diff_dias (REAL), oil_lag_1 (REAL), gas_lag_1 (REAL), water_lag_1 (REAL), oil_lag_3 (REAL), gas_lag_3 (REAL), water_lag_3 (REAL), oil_lag_7 (REAL), gas_lag_7 (REAL), water_lag_7 (REAL), oil_lag_14 (REAL), gas_lag_14 (REAL), water_lag_14 (REAL), oil_lag_30 (REAL), gas_lag_30 (REAL), water_lag_30 (REAL), oil_r